# Sistema Inteligente de Detecção e Classificação de Placas de Trânsito
### AED · Projeto Integrador · 2ª Etapa · Checkpoint 1 (N1)

**Pontifícia Universidade Católica de Goiás** · Escola Politécnica e de Artes
Curso de Ciência de Dados e Inteligência Artificial
Disciplina **CDI1021 · Visão Computacional (2026/2)** · Prof. Welington Júlio Dias Rodrigues

**Equipe:** Caio Henrique · Fernanda Andrade · Alisson Leonardo · Vitor Manoel

---

A partir de uma foto de rua, o notebook gera uma máscara binária das placas e tira dela
contagem, área, centroide, caixa envolvente e classe geométrica. Tudo com PDI clássico,
sem aprendizado de máquina.

| Item do Checkpoint 1 | Onde está |
|---|---|
| Código organizado em funções | Seções 3 a 5 |
| Estrutura do dataset documentada | Seção 2 |
| Correção de iluminação | Seção 3.3 |
| Análise de histograma dos canais | Seção 3.2.1 |
| Limiarização com justificativa técnica | Seções 3.6 e 5.2 |
| Limpeza morfológica | Seções 3.7 e 5.1 |
| Contornos com filtro por área mínima | Seção 3.8 |
| Diagrama da arquitetura | Seção 9 |
| Evidências visuais antes e depois | Seção 7 |
| Registro dos parâmetros adotados | Seção 10 |

**Como rodar:** *Runtime → Executar tudo* (`Ctrl+F9`). A Seção 1 pede a chave gratuita da
API do Roboflow, e tem dois caminhos alternativos na mesma célula. A execução leva uns 25
minutos e termina zipando `outputs/`.

> Nenhum parâmetro foi ajustado até a imagem ficar bonita. Limiar, kernel e área mínima
> saem de medição no próprio dataset (Seção 5), e o método de limiarização é escolhido por
> métrica numa amostra que não é a de avaliação.


---
## 0. Ambiente

Instala só o que não vem pronto no Google Colab. Rodando local, use o `requirements.txt`
do repositório.


In [ ]:
import sys, subprocess, importlib

EM_COLAB = "google.colab" in sys.modules


def garantir_pacote(modulo: str, pacote: str | None = None) -> bool:
    """Importa `modulo`; se ausente, instala `pacote` (padrao: mesmo nome) via pip."""
    try:
        importlib.import_module(modulo)
        return True
    except ImportError:
        alvo = pacote or modulo
        print(f"[setup] instalando {alvo} ...")
        codigo = subprocess.call([sys.executable, "-m", "pip", "install", "-q", alvo])
        if codigo != 0:
            print(f"[setup] FALHA ao instalar {alvo}")
            return False
        importlib.invalidate_caches()
        return True


for mod, pkg in [("cv2", "opencv-python"), ("numpy", "numpy"), ("matplotlib", "matplotlib"),
                 ("pandas", "pandas"), ("yaml", "pyyaml"), ("roboflow", "roboflow")]:
    garantir_pacote(mod, pkg)

print("\nAmbiente:", "Google Colab" if EM_COLAB else "local")

In [ ]:
import json, math, os, random, re, shutil, time, zipfile
from dataclasses import dataclass, field, asdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

def salvar_figura(fig, caminho: Path) -> Path:
    """Grava a figura: JPEG para painel de foto, PNG para grafico (escolhe pela extensao)."""
    caminho = Path(caminho)
    extras = {"pil_kwargs": {"quality": 85, "optimize": True}} if caminho.suffix == ".jpg" else {}
    fig.savefig(caminho, bbox_inches="tight", **extras)
    return caminho


plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 140, "figure.facecolor": "white",
    "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "font.size": 9,
})

# Raiz do projeto: /content no Colab, ou a pasta do notebook se estiver rodando local.
RAIZ = Path("/content/aed_visao") if EM_COLAB else Path.cwd()
if not EM_COLAB and RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

DIR_DADOS = RAIZ / "data"
DIR_SAIDA = RAIZ / "outputs"
DIR_FIGURAS = DIR_SAIDA / "figuras"
DIR_DOCS = RAIZ / "docs"
for d in (DIR_DADOS, DIR_SAIDA, DIR_FIGURAS, DIR_DOCS):
    d.mkdir(parents=True, exist_ok=True)

print(f"OpenCV      {cv2.__version__}")
print(f"NumPy       {np.__version__}")
print(f"pandas      {pd.__version__}")
print(f"Python      {sys.version.split()[0]}")
print(f"\nRaiz do projeto : {RAIZ}")
print(f"Dados           : {DIR_DADOS}")
print(f"Saidas          : {DIR_SAIDA}")

---
## 1. Aquisição do dataset

**Fonte:** [Placas de Trânsito BR](https://universe.roboflow.com/caios-workspace-01wh5/placas-de-transito-br-wq5tp/dataset/1),
cópia em resolução nativa da [base original](https://universe.roboflow.com/stefano-tommasini-coelho-euf67/placas-de-transito-br)
do Roboflow Universe (licença *Public Domain*). São cenas de rodovia brasileira anotadas
com os códigos do CONTRAN (A-1a, R-1, R-19, I-4), o que sustenta a justificativa normativa
das faixas de cor da Seção 3.1.

A cópia existe porque todas as versões publicadas da base original aplicam *Resize to
640×640 (Stretch)*, que espreme a imagem em 2:1 e deforma a geometria das placas. O
efeito medido dessa troca está no README, na Seção 4.

Três caminhos de download, e basta um funcionar:

| Caminho | Quando usar |
|---|---|
| **A · SDK do Roboflow** | O padrão. A chave fica em `roboflow.com` → *Settings* → *API Keys* |
| **B · link direto** | Se a chave falhar. Copie a URL em *Download Dataset* → `YOLOv8` → *show download code* e cole em `URL_DOWNLOAD_DIRETO` |
| **C · pasta local** | Dataset já baixado, ou fotos da própria equipe. Aponte `PASTA_LOCAL` |


In [ ]:
# ------------------------------------------------------------------ configuracao
# Copia da base original no workspace da equipe, gerada SEM o resize para 640x640. O
# export original vinha esticado: 1248x633 comprimido pra 640x640, o que espremia a
# largura quase pela metade e deformava a geometria das placas.
WORKSPACE = "caios-workspace-01wh5"
PROJETO = "placas-de-transito-br-wq5tp"
VERSAO = 1
FORMATO = "yolov8"          # exporta images/ + labels/ (YOLO) + data.yaml

URL_DOWNLOAD_DIRETO = ""    # caminho B: cole aqui a URL "https://universe.roboflow.com/ds/...?key=..."
PASTA_LOCAL = ""            # caminho C: cole aqui o caminho de uma pasta ja baixada

# Tamanho das amostras de calibracao e avaliacao. Segura o tempo de execucao.
N_AMOSTRA_CALIBRACAO = 400  # imagens lidas para estimar escala do objeto
N_AMOSTRA_VALIDACAO = 250   # imagens por amostra (ajuste e validacao, disjuntas)
N_EVIDENCIAS_VISUAIS = 6    # imagens com painel antes/depois (o enunciado exige >= 3)

In [ ]:
def baixar_via_sdk(destino: Path) -> Path | None:
    """Caminho A: baixa a versao publica do Roboflow Universe usando o SDK oficial."""
    try:
        from getpass import getpass
        from roboflow import Roboflow
    except ImportError:
        print("[dados] pacote roboflow indisponivel")
        return None
    chave = os.environ.get("ROBOFLOW_API_KEY") or getpass("Chave da API do Roboflow: ").strip()
    if not chave:
        print("[dados] nenhuma chave informada")
        return None
    try:
        rf = Roboflow(api_key=chave)
        projeto = rf.workspace(WORKSPACE).project(PROJETO)
        conjunto = projeto.version(VERSAO).download(FORMATO, location=str(destino))
        return Path(conjunto.location)
    except Exception as erro:                                  # noqa: BLE001
        print(f"[dados] SDK falhou: {type(erro).__name__}: {erro}")
        return None


def baixar_via_url(url: str, destino: Path) -> Path | None:
    """Caminho B: baixa e extrai o zip gerado pelo botao *Download Dataset*."""
    if not url:
        return None
    destino.mkdir(parents=True, exist_ok=True)
    zip_local = destino / "roboflow.zip"
    print("[dados] baixando zip ...")
    codigo = subprocess.call(["curl", "-sSL", "-o", str(zip_local), url])
    if codigo != 0 or not zip_local.exists():
        print("[dados] download direto falhou")
        return None
    with zipfile.ZipFile(zip_local) as z:
        z.extractall(destino)
    zip_local.unlink(missing_ok=True)
    return destino


def localizar_dataset(raiz: Path) -> Path | None:
    """Procura recursivamente a pasta que contem `data.yaml` ou, na falta dele, imagens."""
    raiz = Path(raiz)
    if not raiz.exists():
        return None
    candidatos = sorted(raiz.rglob("data.yaml"))
    if candidatos:
        return candidatos[0].parent
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        achado = next(iter(raiz.rglob(ext)), None)
        if achado is not None:
            return raiz
    return None


def obter_dataset() -> Path:
    """Resolve a origem do dataset seguindo a ordem C -> ja baixado -> A -> B."""
    if PASTA_LOCAL:
        achado = localizar_dataset(Path(PASTA_LOCAL))
        if achado:
            print(f"[dados] usando pasta local: {achado}")
            return achado
        raise FileNotFoundError(f"PASTA_LOCAL nao contem imagens: {PASTA_LOCAL}")

    achado = localizar_dataset(DIR_DADOS)
    if achado:
        print(f"[dados] dataset ja presente em: {achado}")
        return achado

    alvo = DIR_DADOS / f"{PROJETO}-{VERSAO}"
    for tentativa in (lambda: baixar_via_sdk(alvo),
                      lambda: baixar_via_url(URL_DOWNLOAD_DIRETO, alvo)):
        resultado = tentativa()
        if resultado:
            achado = localizar_dataset(resultado) or localizar_dataset(DIR_DADOS)
            if achado:
                print(f"[dados] dataset disponivel em: {achado}")
                return achado

    raise FileNotFoundError(
        "Nao foi possivel obter o dataset.\n"
        "  - Caminho A: informe uma chave valida da API do Roboflow;\n"
        "  - Caminho B: preencha URL_DOWNLOAD_DIRETO na celula de configuracao;\n"
        "  - Caminho C: preencha PASTA_LOCAL com uma pasta que contenha as imagens."
    )

In [ ]:
DIR_DATASET = obter_dataset()

ARQ_YAML = DIR_DATASET / "data.yaml"
if ARQ_YAML.exists():
    META = yaml.safe_load(ARQ_YAML.read_text(encoding="utf-8"))
    CLASSES = list(META.get("names", []) or [])
    print(f"data.yaml encontrado | {len(CLASSES)} classes anotadas")
    print("Primeiras classes:", CLASSES[:8])
else:
    META, CLASSES = {}, []
    print("Aviso: data.yaml ausente. O inventario segue, mas sem os nomes das classes.")

---
## 2. Inventário e caracterização do dataset

O Checkpoint 1 pede a estrutura do dataset documentada. O inventário abaixo é gerado
automaticamente e salvo em `outputs/inventario_dataset.csv`, então o número que aparece no
README é sempre o da base realmente usada.

```
<dataset>/
├── data.yaml            # nomes das classes e caminhos dos splits
├── train/  images/  labels/
├── valid/  images/  labels/
└── test/   images/  labels/
```

Cada `labels/<nome>.txt` tem uma linha por objeto: `classe cx cy largura altura`,
normalizados.

**Detalhe desta base:** 5 linhas (em 3 arquivos) vêm como polígono, com pares `x y` em vez
de caixa. Lidas do jeito normal virariam placas com 75% da largura da imagem, então o
`ler_rotulos` converte o polígono na caixa que o envolve.


In [ ]:
EXTENSOES = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def listar_imagens(raiz: Path) -> list[Path]:
    """Imagens sob `raiz`, na mesma ordem em qualquer sistema.

    Ordena pelo caminho POSIX em minusculas, senao Windows e Linux dariam amostras
    diferentes para a mesma semente.
    """
    return sorted((p for p in Path(raiz).rglob("*") if p.suffix.lower() in EXTENSOES),
                  key=lambda p: p.as_posix().lower())


def caminho_rotulo(img: Path) -> Path:
    """Caminho do .txt YOLO correspondente a uma imagem (troca images/ por labels/)."""
    partes = list(img.parts)
    for i in range(len(partes) - 1, -1, -1):
        if partes[i] == "images":
            partes[i] = "labels"
            break
    return Path(*partes).with_suffix(".txt")


def ler_rotulos(img: Path) -> list[tuple[int, float, float, float, float]]:
    """Anotacoes YOLO normalizadas: (classe, cx, cy, w, h). Vazio se nao houver.

    A base mistura caixa (4 valores) e poligono (pares x y); o poligono vira a caixa que
    o envolve, senao coordenadas de ponto virariam largura e altura.
    """
    arq = caminho_rotulo(img)
    if not arq.exists():
        return []
    itens = []
    for linha in arq.read_text(encoding="utf-8", errors="ignore").splitlines():
        campos = linha.split()
        if len(campos) < 5:
            continue
        try:
            classe = int(float(campos[0]))
            valores = [float(v) for v in campos[1:]]
        except ValueError:
            continue
        if len(valores) == 4:
            itens.append((classe, *valores))
        elif len(valores) >= 6 and len(valores) % 2 == 0:
            xs, ys = valores[0::2], valores[1::2]
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            itens.append((classe, (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0))
    return itens


def dimensoes(img: Path) -> tuple[int, int]:
    """(largura, altura) reais da imagem. (0, 0) se o arquivo nao decodificar."""
    dados = np.fromfile(str(img), dtype=np.uint8)
    mat = cv2.imdecode(dados, cv2.IMREAD_COLOR)
    if mat is None:
        return (0, 0)
    return (mat.shape[1], mat.shape[0])


def inventariar(raiz: Path, amostra_dimensoes: int = 400) -> pd.DataFrame:
    """Monta o inventario: split, arquivo, dimensoes e numero de objetos anotados."""
    imagens = listar_imagens(raiz)
    if not imagens:
        raise FileNotFoundError(f"Nenhuma imagem encontrada em {raiz}")
    passo = max(1, len(imagens) // amostra_dimensoes)
    linhas = []
    for i, img in enumerate(imagens):
        rotulos = ler_rotulos(img)
        partes = set(img.parts)
        split = next((s for s in ("train", "valid", "val", "test") if s in partes), "unico")
        larg, alt = dimensoes(img) if i % passo == 0 else (np.nan, np.nan)
        linhas.append({
            "arquivo": img.name, "caminho": str(img), "split": split,
            "largura": larg, "altura": alt,
            "n_objetos": len(rotulos),
            "classes": ",".join(sorted({str(r[0]) for r in rotulos})),
        })
    return pd.DataFrame(linhas)

In [ ]:
INVENTARIO = inventariar(DIR_DATASET)
INVENTARIO.to_csv(DIR_SAIDA / "inventario_dataset.csv", index=False, encoding="utf-8")

dim = INVENTARIO.dropna(subset=["largura"])
resumo_split = (INVENTARIO.groupby("split")
                .agg(imagens=("arquivo", "size"),
                     objetos=("n_objetos", "sum"),
                     objetos_por_imagem=("n_objetos", "mean"))
                .round(2))

print("=" * 66)
print("INVENTARIO DO DATASET")
print("=" * 66)
print(f"Fonte            : Roboflow Universe / {WORKSPACE}/{PROJETO} v{VERSAO}")
print(f"Local            : {DIR_DATASET}")
print(f"Imagens          : {len(INVENTARIO)}")
print(f"Objetos anotados : {int(INVENTARIO['n_objetos'].sum())}")
print(f"Classes          : {len(CLASSES)}")
if len(dim):
    print(f"Dimensoes        : {int(dim['largura'].median())} x {int(dim['altura'].median())} px "
          f"(mediana de {len(dim)} imagens medidas)")
    print(f"  minimo         : {int(dim['largura'].min())} x {int(dim['altura'].min())} px")
    print(f"  maximo         : {int(dim['largura'].max())} x {int(dim['altura'].max())} px")
print(f"Sem anotacao     : {int((INVENTARIO['n_objetos'] == 0).sum())} imagens")
print("=" * 66)
resumo_split

### 2.1. Escala do objeto na cena

É esta medida que dimensiona os kernels e a área mínima (Seção 5.1), em vez de escolher
`5×5` por hábito. A fração de área também diagnostica a base: mediana alta indicaria
recortes centralizados na placa, e não cenas completas.

Aqui a medida é descritiva, sobre a base inteira. Na calibração ela é refeita usando só o
lado de ajuste, para as anotações de validação não entrarem em nenhum parâmetro.


In [ ]:
def estatisticas_de_escala(inventario: pd.DataFrame, n: int, largura_trabalho: int) -> pd.DataFrame:
    """Tamanho das caixas anotadas reescalado para a largura de trabalho do pipeline."""
    com_objeto = inventario[inventario["n_objetos"] > 0]
    if com_objeto.empty:
        return pd.DataFrame()
    amostra = com_objeto.sample(min(n, len(com_objeto)), random_state=SEMENTE)
    linhas = []
    for caminho in amostra["caminho"]:
        img = Path(caminho)
        rotulos = ler_rotulos(img)
        if not rotulos:
            continue
        larg, alt = dimensoes(img)
        if not larg or not alt:
            continue
        escala = largura_trabalho / larg
        alt_trab = alt * escala
        for _, _, _, w, h in rotulos:
            w_px, h_px = w * largura_trabalho, h * alt_trab
            linhas.append({
                "largura_px": w_px, "altura_px": h_px,
                "area_px": w_px * h_px,
                "lado_equivalente_px": math.sqrt(max(w_px * h_px, 1e-9)),
                "fracao_area": w * h,
            })
    return pd.DataFrame(linhas)


# Largura nativa das imagens. Redimensionar pra menos encolheria os objetos, que ja sao
# pequenos; o pipeline preserva a proporcao, entao a altura acompanha.
LARGURA_TRABALHO = 1248
ESCALA = estatisticas_de_escala(INVENTARIO, N_AMOSTRA_CALIBRACAO, LARGURA_TRABALHO)

if ESCALA.empty:
    print("Sem anotacoes: os parametros usarao os valores padrao da Secao 4.")
else:
    q = ESCALA[["lado_equivalente_px", "area_px", "fracao_area"]].describe(
        percentiles=[0.05, 0.10, 0.25, 0.5, 0.75, 0.95]).round(2)
    print(f"{len(ESCALA)} caixas anotadas medidas, reescaladas para largura {LARGURA_TRABALHO} px\n")
    print(q.to_string())
    mediana_fracao = ESCALA["fracao_area"].median()
    print(f"\nA placa mediana ocupa {mediana_fracao * 100:.2f}% da area da imagem.")
    if mediana_fracao > 0.35:
        print("DIAGNOSTICO: a base e dominada por recortes proximos da placa, nao por cenas completas.")
    else:
        print("DIAGNOSTICO: cenas completas, entao ha mesmo segmentacao a ser feita.")

In [ ]:
if not ESCALA.empty:
    fig, eixos = plt.subplots(1, 3, figsize=(13, 3.4))
    eixos[0].hist(ESCALA["lado_equivalente_px"], bins=40, color="#2d6a9f", edgecolor="white")
    eixos[0].axvline(ESCALA["lado_equivalente_px"].median(), color="#c1440e", lw=2,
                     label=f"mediana {ESCALA['lado_equivalente_px'].median():.0f} px")
    eixos[0].set_title("Lado equivalente da placa"); eixos[0].set_xlabel("px"); eixos[0].legend()

    eixos[1].hist(ESCALA["area_px"], bins=40, color="#2d6a9f", edgecolor="white")
    p5 = ESCALA["area_px"].quantile(0.05)
    eixos[1].axvline(p5, color="#c1440e", lw=2, label=f"percentil 5 = {p5:.0f} px²")
    eixos[1].set_title("Área da caixa anotada"); eixos[1].set_xlabel("px²")
    eixos[1].set_yscale("log"); eixos[1].legend()

    eixos[2].hist(ESCALA["fracao_area"] * 100, bins=40, color="#2d6a9f", edgecolor="white")
    eixos[2].set_title("Fração da imagem ocupada"); eixos[2].set_xlabel("%")
    eixos[2].set_yscale("log")

    fig.suptitle("Escala do objeto no dataset (base para dimensionar os parâmetros)", y=1.04)
    fig.tight_layout()
    fig.savefig(DIR_FIGURAS / "00_escala_do_objeto.png", bbox_inches="tight")
    plt.show()

---
## 3. O pipeline, etapa por etapa

Cada etapa é uma função independente. O orquestrador da Seção 4 só encadeia:

```
imagem → redimensionar → corrigir iluminação → suavizar → mapa de evidência cromática
       → limiarizar → morfologia → contornos → descritores
```

Três decisões de ordem:

1. **Suavizar antes de limiarizar.** Ruído de alta frequência polui o histograma e desloca
   o limiar de Otsu. A Seção 6 mede o tamanho desse efeito.
2. **Corrigir iluminação antes de converter para HSV.** Contraluz e sombra derrubam o `V`,
   e o CLAHE reequilibra a luminância sem mexer na matiz.
3. **Contornos saem da máscara morfológica, nunca do Canny.** Borda de um pixel tem dois
   lados, e o `findContours` devolveria dois contornos por objeto, dobrando a contagem.


### 3.1. Parâmetros e faixas normativas de cor

`Parametros` junta tudo que governa o pipeline. Os valores aqui são ponto de partida: a
Seção 5 sobrescreve kernels, área mínima, portas de saturação e método por medição.

As âncoras de matiz vêm das cores normativas da sinalização vertical brasileira (Resolução
CONTRAN nº 160/2004 e MBST):

| Cor | Uso normativo | Âncora (H do OpenCV, 0 a 179) |
|---|---|---|
| Vermelho | Regulamentação (R-1, R-19, orlas) | 0, e a matiz é circular, envolve 0 e 179 |
| Amarelo | Advertência (A-*) e delineadores | 20 |
| Verde | Indicação e destino (I-4) | 65 |
| Azul | Serviços auxiliares (S-14, LOC-6) | 108 |

**São só âncoras.** A Seção 5.0 mede a matiz dentro das caixas anotadas e recalcula centro
e largura de cada faixa. Âncora sem objeto anotado suficiente não vira faixa.

Cada faixa tem duas portas, `s_min` e `v_min`, que descartam pixel acinzentado ou escuro
demais para ter matiz confiável. A porta de saturação é o parâmetro que mais decide quanto
de fundo entra na máscara, e o estágio 0 da busca (Seção 5.2) pode até descartar a faixa
inteira.


In [ ]:
# Ancoras de matiz das cores normativas, na escala H do OpenCV (0..179). O centro e a
# largura sao recalculados na Secao 5.0 a partir das anotacoes, e o `s_min` sai por
# metrica no estagio 0 da Secao 5.2. O `v_min` joga fora pixel escuro demais pra ter
# matiz confiavel.
FAIXAS_CONTRAN = [
    {"nome": "vermelho", "h_centro": 4.0,   "h_sigma": 10.0, "s_min": 100, "v_min": 50},
    {"nome": "amarelo",  "h_centro": 20.0,  "h_sigma": 8.0,  "s_min": 120, "v_min": 80},
    {"nome": "verde",    "h_centro": 65.0,  "h_sigma": 14.0, "s_min": 110, "v_min": 60},
    {"nome": "azul",     "h_centro": 108.0, "h_sigma": 9.0,  "s_min": 110, "v_min": 50},
]


def faixas_com_portas(faixas: list, portas: dict) -> list:
    """Copia de `faixas` com outras portas de saturacao; `None` remove a faixa."""
    novas = []
    for faixa in faixas:
        if faixa["nome"] in portas:
            if portas[faixa["nome"]] is None:
                continue
            novas.append({**faixa, "s_min": int(portas[faixa["nome"]])})
        else:
            novas.append(dict(faixa))
    return novas


@dataclass
class Parametros:
    """Configuracao completa de uma execucao do pipeline."""
    # entrada
    largura_trabalho: int = 1248
    # etapa A - iluminacao
    usar_clahe: bool = True
    clahe_clip: float = 2.0
    clahe_grade: int = 8
    # etapa B - suavizacao (o kernel e derivado da escala do objeto na Secao 5.1)
    suavizacao: str = "gaussiano"        # "gaussiano" | "mediana" | "bilateral" | "nenhuma"
    suavizacao_k: int = 3
    # etapa C - evidencia cromatica
    faixas: list = field(default_factory=lambda: [dict(f) for f in FAIXAS_CONTRAN])
    # etapa D - limiarizacao
    metodo_limiar: str = "otsu_restrito"  # "global" | "otsu" | "otsu_restrito" | "adaptativa"
    # Valores fixos: a busca em grade escolhe o metodo, mas nao ajusta estes tres.
    limiar_global: int = 96               # limiar do metodo global
    adapt_bloco: int = 51                 # blockSize da adaptativa (janela em px, impar)
    adapt_c: int = -10                    # C da adaptativa (subtraido da media local)
    # etapa E - morfologia
    k_abertura: int = 3
    k_fechamento: int = 11
    preencher_buracos: bool = True
    # etapa F - contornos e filtros de forma
    area_minima: int = 200
    area_maxima: int = 0                  # 0 = sem limite superior
    razao_aspecto: tuple = (0.35, 2.85)
    extensao_minima: float = 0.35
    solidez_minima: float = 0.70

    def para_dict(self) -> dict:
        d = asdict(self)
        d["razao_aspecto"] = list(self.razao_aspecto)
        return d


def impar(n, minimo: int = 3) -> int:
    """Arredonda para o inteiro impar >= `minimo` (kernels do OpenCV exigem lado impar)."""
    n = int(round(n))
    n = max(n, minimo)
    return n if n % 2 == 1 else n + 1

### 3.2. Entrada e redimensionamento

A largura de trabalho é 1.248 px, a resolução nativa da base. Padronizar a largura deixa
os parâmetros em pixel comparáveis entre imagens: um kernel de 11 px significa coisas
diferentes em 640 e em 1920 px.

`carregar_imagem` usa `np.fromfile` + `cv2.imdecode` em vez de `cv2.imread`, que falha
calado quando o caminho tem acento ou espaço.


In [ ]:
def carregar_imagem(caminho) -> np.ndarray:
    """Le uma imagem do disco e devolve um array RGB uint8."""
    dados = np.fromfile(str(caminho), dtype=np.uint8)
    bgr = cv2.imdecode(dados, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Nao foi possivel decodificar a imagem: {caminho}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def redimensionar(rgb: np.ndarray, largura: int = 640) -> tuple[np.ndarray, float]:
    """Reescala preservando a proporcao. Devolve (imagem, fator de escala aplicado)."""
    altura_orig, largura_orig = rgb.shape[:2]
    if largura_orig == largura:
        return rgb.copy(), 1.0
    escala = largura / float(largura_orig)
    novo = (largura, max(1, int(round(altura_orig * escala))))
    # INTER_AREA e o reamostrador correto para reducao: media os pixels da vizinhanca
    # em vez de amostra-los, o que evita aliasing nas bordas da placa.
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_LINEAR
    return cv2.resize(rgb, novo, interpolation=interp), escala

### 3.3. Etapa A · Correção de iluminação

**CLAHE**, aplicado só no canal `L` do LAB. Equalizar os três canais RGB mudaria a
proporção entre eles e deslocaria a matiz, que é o atributo em que a segmentação se apoia.
No LAB o `L` carrega a luminância e o `a`/`b` a cromaticidade, então corrigir só o `L`
arruma a luz e preserva a cor.

O `clipLimit = 2.0` limita a amplificação por bloco; acima de 4 satura o céu e realça
ruído. A grade `8×8` dá blocos de cerca de 156×80 px, algumas vezes maior que a placa
típica, grande o suficiente para capturar a variação local de luz sem tratar a placa como
fundo.

**Top-hat** (imagem menos sua abertura) isola estrutura clara menor que o elemento e
remove gradiente lento de fundo. Fica como alternativa e aparece na comparação visual, mas
fora do pipeline padrão: ele exige tons de cinza, e aí a cor iria embora.


In [ ]:
def corrigir_iluminacao_clahe(rgb: np.ndarray, clip: float = 2.0, grade: int = 8) -> np.ndarray:
    """CLAHE so no canal L do LAB: corrige a luminancia sem mexer na matiz."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    canal_l, canal_a, canal_b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(grade), int(grade)))
    lab_corrigido = cv2.merge([clahe.apply(canal_l), canal_a, canal_b])
    return cv2.cvtColor(lab_corrigido, cv2.COLOR_LAB2RGB)


def corrigir_iluminacao_tophat(rgb: np.ndarray, k: int = 25) -> np.ndarray:
    """Top-hat em tons de cinza: realca o que e claro e menor que o elemento."""
    cinza = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    elemento = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (impar(k), impar(k)))
    return cv2.morphologyEx(cinza, cv2.MORPH_TOPHAT, elemento)

### 3.2.1. Análise de histograma dos canais

O enunciado pede conversão **e análise** de espaços de cor. Esta célula mostra a
distribuição dos pixels em RGB, tons de cinza e HSV, antes e depois do CLAHE, em escala
logarítmica (senão o pico do fundo achata o resto).

Ela responde duas perguntas com número:

- **O que o CLAHE faz com a luminância?** Espalha o histograma do `V`, que é o efeito
  esperado de uma equalização adaptativa.
- **Ele mexe na cor?** O deslocamento de matiz é medido duas vezes, em todos os pixels e
  só nos saturados. Pixel acinzentado tem matiz instável por definição, e já é descartado
  pela porta de saturação; o número que vale é o dos saturados. É a justificativa numérica
  de corrigir o `L*` em vez de equalizar RGB.


In [ ]:
def histogramas_dos_canais(caminho: Path, params: Parametros,
                           salvar: Path | None = None):
    """Distribuicao de cada canal antes e depois do CLAHE, em RGB, cinza e HSV.

    Fora do pipeline: e a leitura que sustenta corrigir a luminancia sem mexer na matiz.
    """
    original, _ = redimensionar(carregar_imagem(caminho), params.largura_trabalho)
    corrigida = corrigir_iluminacao_clahe(original, params.clahe_clip, params.clahe_grade)

    fig, eixos = plt.subplots(2, 4, figsize=(16, 6.4))
    for linha, (nome, imagem) in enumerate([("antes", original), ("depois do CLAHE", corrigida)]):
        cinza = cv2.cvtColor(imagem, cv2.COLOR_RGB2GRAY)
        hsv = cv2.cvtColor(imagem, cv2.COLOR_RGB2HSV)

        eixos[linha, 0].imshow(imagem)
        eixos[linha, 0].set_title(f"Imagem {nome}", fontsize=9)
        eixos[linha, 0].axis("off")

        for canal, cor in zip(range(3), ("#c0392b", "#27ae60", "#2d6a9f")):
            valores = cv2.calcHist([imagem], [canal], None, [256], [0, 256]).ravel()
            eixos[linha, 1].plot(valores, color=cor, lw=1.2, label="RGB"[canal])
        eixos[linha, 1].set_title(f"RGB ({nome})", fontsize=9)
        eixos[linha, 1].legend(fontsize=7)

        eixos[linha, 2].plot(cv2.calcHist([cinza], [0], None, [256], [0, 256]).ravel(),
                             color="#34495e", lw=1.2)
        eixos[linha, 2].set_title(f"Tons de cinza ({nome})", fontsize=9)

        for canal, cor, rotulo, limite in [(0, "#8e44ad", "H", 180), (1, "#e08214", "S", 256),
                                           (2, "#16a085", "V", 256)]:
            valores = cv2.calcHist([hsv], [canal], None, [limite], [0, limite]).ravel()
            eixos[linha, 3].plot(valores, color=cor, lw=1.2, label=rotulo)
        eixos[linha, 3].set_title(f"HSV ({nome})", fontsize=9)
        eixos[linha, 3].legend(fontsize=7)

        for coluna in (1, 2, 3):
            eixos[linha, coluna].set_yscale("log")
            eixos[linha, coluna].grid(alpha=0.3)
            eixos[linha, coluna].set_xlabel("intensidade", fontsize=8)

    fig.suptitle(f"Histogramas por canal · {caminho.name[:44]}", y=1.0)
    fig.tight_layout()
    if salvar:
        salvar_figura(fig, salvar)

    # O que a figura mostra, em numero. O deslocamento de matiz e medido duas vezes: em
    # todos os pixels e so nos saturados. Pixel acinzentado tem matiz instavel por
    # definicao (com S baixo, qualquer mudancinha em R, G ou B gira o angulo da cor), e
    # esse pixel ja e descartado pela porta de saturacao do pipeline. O numero que importa
    # e o dos pixels saturados, que sao os que a segmentacao enxerga.
    hsv_antes = cv2.cvtColor(original, cv2.COLOR_RGB2HSV).astype(np.int16)
    hsv_depois = cv2.cvtColor(corrigida, cv2.COLOR_RGB2HSV).astype(np.int16)
    diferenca = np.abs(hsv_depois[:, :, 0] - hsv_antes[:, :, 0])
    desvio_h = np.minimum(diferenca, 180 - diferenca)          # matiz e circular
    saturados = hsv_antes[:, :, 1] >= 100                      # ordem da porta adotada
    return {
        "v_medio_antes": round(float(hsv_antes[:, :, 2].mean()), 1),
        "v_medio_depois": round(float(hsv_depois[:, :, 2].mean()), 1),
        "v_desvio_antes": round(float(hsv_antes[:, :, 2].std()), 1),
        "v_desvio_depois": round(float(hsv_depois[:, :, 2].std()), 1),
        "matiz_deslocada_todos_os_pixels": round(float(desvio_h.mean()), 2),
        "matiz_deslocada_pixels_saturados": round(float(desvio_h[saturados].mean()), 2)
        if saturados.any() else float("nan"),
        "fracao_de_pixels_saturados": round(float(saturados.mean()), 3),
    }


_com_objeto = INVENTARIO[INVENTARIO["n_objetos"] > 0]
IMAGEM_HISTOGRAMA = Path(_com_objeto.sample(1, random_state=SEMENTE)["caminho"].iloc[0])
EFEITO_CLAHE = histogramas_dos_canais(IMAGEM_HISTOGRAMA, Parametros(),
                                      DIR_FIGURAS / "00d_histogramas_canais.jpg")
plt.show()

print("EFEITO DO CLAHE NESTA IMAGEM")
print("-" * 58)
for chave, valor in EFEITO_CLAHE.items():
    print(f"{chave:<36} {valor}")
print("-" * 58)
print("Leitura: o CLAHE mexe na luminancia, que e o que se quer. A matiz se desloca pouco\n"
      "nos pixels saturados, que sao os unicos que a segmentacao usa; nos acinzentados ela\n"
      "gira mais, mas esses ja caem na porta de saturacao. E por isso que a correcao e feita\n"
      "no L* do LAB: equalizar os tres canais RGB mudaria a proporcao entre eles e levaria\n"
      "a matiz junto, inclusive a das placas.")


### 3.4. Etapa B · Suavização

O **gaussiano** é o padrão. Atenua o ruído de sensor e a textura fina de asfalto e
vegetação, que são o que mais fragmenta a máscara, antes que isso contamine o histograma
da limiarização.

O tamanho não foi escolhido por hábito. A estrutura mais fina a preservar é a orla da placa
pequena, com 1 a 2 px de espessura, e qualquer borrão mais largo mistura orla e miolo. Por
isso o kernel sai da mesma regra da abertura, `ímpar(0,10 × p10 do lado da placa)`,
calculada na Seção 5.1.

A **mediana** fica disponível para ruído impulsivo, e o **bilateral** para suavizar
preservando borda.

> A ablação da Seção 6 compara gaussiano, mediana e nenhuma suavização. A diferença é
> pequena, porque o export já passou por reamostragem e recompressão, que atenuam o ruído
> de sensor. O gaussiano fica porque é etapa exigida pelo Checkpoint 1 e porque as fotos
> que a equipe vai tirar chegam sem esse pré-tratamento.


In [ ]:
def suavizar(rgb: np.ndarray, metodo: str = "gaussiano", k: int = 5) -> np.ndarray:
    """Tira ruido antes de limiarizar."""
    if metodo == "nenhuma":
        return rgb.copy()
    k = impar(k)
    if metodo == "gaussiano":
        return cv2.GaussianBlur(rgb, (k, k), 0)
    if metodo == "mediana":
        return cv2.medianBlur(rgb, k)
    if metodo == "bilateral":
        return cv2.bilateralFilter(rgb, k, 75, 75)
    raise ValueError(f"metodo de suavizacao desconhecido: {metodo}")

### 3.5. Etapa C · Mapa de evidência cromática

É a etapa que transforma um problema de cor num problema de limiarização.

Limiarizar a imagem em tons de cinza não funciona: placa vermelha e asfalto podem ter a
mesma luminância. O que distingue a placa é matiz normativa com saturação alta. Em vez de
uma máscara binária por faixa de HSV, que exige bordas rígidas, o pipeline constrói um mapa
escalar contínuo em `[0, 255]`:

$$E(x,y) \;=\; \underbrace{\max_{c \,\in\, \{\text{verm.},\,\text{amar.}\}} \exp\!\left(-\frac{d_H(H_{x,y},\,\mu_c)^2}{2\sigma_c^2}\right) \cdot \mathbb{1}\!\left[S_{x,y} \ge S_c^{\min} \wedge V_{x,y} \ge V_c^{\min}\right]}_{\text{proximidade da matiz normativa}} \;\times\; \underbrace{\frac{S_{x,y}}{255}}_{\text{pureza da cor}}$$

- **`d_H` é distância circular.** O vermelho ocupa as duas pontas da escala `H`; tratar `H`
  como eixo linear partiria a placa vermelha no meio.
- **O peso gaussiano evita borda rígida.** Matiz um pouco desviada recebe peso menor, e
  quem decide é a limiarização.
- **A porta `S_min` é a que mais decide.** Grama seca, solo exposto e fachada têm matiz
  parecida com saturação média, e só uma porta alta separa isso da película da placa.

A saída é um canal só, com o objeto na cauda alta do histograma.


In [ ]:
def distancia_circular_matiz(h: np.ndarray, centro: float, periodo: float = 180.0) -> np.ndarray:
    """Distancia circular entre matizes: o vermelho fica nas duas pontas da escala H."""
    d = np.abs(h.astype(np.float32) - float(centro))
    return np.minimum(d, periodo - d)


def mapa_evidencia_cromatica(rgb: np.ndarray, faixas: list | None = None) -> np.ndarray:
    """Mapa escalar uint8: quanto cada pixel se parece com a cor de uma placa."""
    faixas = faixas if faixas is not None else FAIXAS_CONTRAN
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    canal_h, canal_s, canal_v = cv2.split(hsv)

    peso = np.zeros(canal_h.shape, dtype=np.float32)
    for faixa in faixas:
        d = distancia_circular_matiz(canal_h, faixa["h_centro"])
        proximidade = np.exp(-(d ** 2) / (2.0 * float(faixa["h_sigma"]) ** 2))
        porta = ((canal_s >= faixa["s_min"]) & (canal_v >= faixa["v_min"])).astype(np.float32)
        peso = np.maximum(peso, proximidade * porta)

    mapa = peso * (canal_s.astype(np.float32) / 255.0)
    return np.clip(mapa * 255.0, 0, 255).astype(np.uint8)

### 3.6. Etapa D · Limiarização

Quatro métodos implementados, comparados na busca em grade da Seção 5.2:

| Método | Como decide o limiar | O que esperar aqui |
|---|---|---|
| **Global fixo** | Constante escolhida antes | Reprodutível e barato, mas cego à imagem |
| **Otsu** | Maximiza a variância entre classes no histograma inteiro | Pressupõe histograma bimodal. Aqui o fundo domina e o limiar tende a cair demais |
| **Otsu restrito** | Mesmo critério, só sobre os pixels com evidência > 0 | Descarta a massa de zeros e devolve a bimodalidade que o método precisa |
| **Adaptativa** | Limiar por vizinhança gaussiana local | Aguenta gradiente de luz, mas realça ruído local em fundo uniforme |

**Nesta base os quatro empatam**, dentro de 0,005 de F1, que é menos que o ruído de uma
amostra de 250 imagens. Deixar o vencedor sair da ordem da tabela seria sorte, então o
desempate é declarado: entre empatados fica o mais simples, e `METODOS_LIMIAR` está escrita
do mais simples ao mais complexo. Isso adota a **limiarização global**, com o corte
escolhido no estágio 1b.

O custo fica registrado: corte fixo funciona nesta base, mas não transfere. Com outra
câmera, é o primeiro valor a recalibrar.

**O T da Apostila.** A receita da Apostila 02 imprime o limiar T de cada imagem. Como o
método adotado usa corte fixo, a Seção 7 calcula o T do Otsu restrito sobre o mesmo mapa,
só como diagnóstico de iluminação. Ele não mexe na máscara.

`_otsu_1d` reimplementa Otsu em NumPy porque `cv2.threshold` só aplica ao histograma da
imagem inteira, sem como restringir a um subconjunto de pixels.


In [ ]:
def _otsu_1d(valores: np.ndarray) -> int:
    """Limiar de Otsu sobre um vetor uint8 qualquer (variancia entre classes maxima)."""
    hist = np.bincount(valores.ravel(), minlength=256).astype(np.float64)
    total = hist.sum()
    if total == 0:
        return 127
    prob = hist / total
    niveis = np.arange(256)
    peso = np.cumsum(prob)                      # w(t)
    media = np.cumsum(prob * niveis)            # mu(t)
    media_total = media[-1]                     # mu_T
    denominador = peso * (1.0 - peso)
    with np.errstate(divide="ignore", invalid="ignore"):
        variancia_entre = (media_total * peso - media) ** 2 / denominador
    variancia_entre[~np.isfinite(variancia_entre)] = -1.0
    return int(np.argmax(variancia_entre))


def limiarizar(mapa: np.ndarray, metodo: str = "otsu_restrito", limiar_global: int = 96,
               adapt_bloco: int = 51, adapt_c: int = -10) -> tuple[np.ndarray, float]:
    """Binariza o mapa de evidencia. Devolve (mascara, limiar aplicado)."""
    if metodo == "global":
        limiar, mascara = cv2.threshold(mapa, int(limiar_global), 255, cv2.THRESH_BINARY)
        return mascara, float(limiar)

    if metodo == "otsu":
        limiar, mascara = cv2.threshold(mapa, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return mascara, float(limiar)

    if metodo == "otsu_restrito":
        # Restringe o histograma aos pixels com alguma evidencia cromatica: sem isso,
        # a moda gigante de zeros do fundo domina a variancia e derruba o limiar.
        candidatos = mapa[mapa > 0]
        if candidatos.size < 50:
            return np.zeros_like(mapa), 255.0
        limiar = float(_otsu_1d(candidatos))
        return ((mapa > limiar) * 255).astype(np.uint8), limiar

    if metodo == "adaptativa":
        mascara = cv2.adaptiveThreshold(
            mapa, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
            impar(adapt_bloco), int(adapt_c))
        return mascara, float("nan")           # limiar e local, nao ha valor unico

    raise ValueError(f"metodo de limiarizacao desconhecido: {metodo}")

### 3.7. Etapa E · Limpeza morfológica

**Abertura** (erosão e depois dilatação), kernel pequeno: tira fragmento isolado de poucos
pixels, como reflexo em lataria e luz de freio. Precisa ser menor que a menor placa que se
quer detectar.

**Fechamento** (dilatação e depois erosão), kernel maior: é a operação essencial. A placa
de regulamentação é uma orla vermelha em volta de um miolo branco, e o mapa de evidência
enxerga o anel, não o disco. Sem fechar, o `findContours` devolveria um anel fino, com área
e centroide errados.

**Preenchimento de buracos:** redesenha cada contorno externo preenchido, para que área,
solidez e extensão descrevam a placa inteira.

Os dois kernels saem da escala real do objeto (Seção 5.1).


In [ ]:
def preencher_buracos(mascara: np.ndarray) -> np.ndarray:
    """Redesenha cada contorno externo preenchido, eliminando vazios internos."""
    contornos = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    cheia = np.zeros_like(mascara)
    if contornos:
        cv2.drawContours(cheia, contornos, -1, 255, thickness=cv2.FILLED)
    return cheia


def limpar_mascara(mascara: np.ndarray, k_abertura: int = 3, k_fechamento: int = 11,
                   preencher: bool = True) -> np.ndarray:
    """Abertura (remove ruido) -> fechamento (consolida a orla) -> preenchimento."""
    saida = mascara.copy()
    if k_abertura and k_abertura >= 3:
        elemento = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (impar(k_abertura),) * 2)
        saida = cv2.morphologyEx(saida, cv2.MORPH_OPEN, elemento)
    if k_fechamento and k_fechamento >= 3:
        elemento = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (impar(k_fechamento),) * 2)
        saida = cv2.morphologyEx(saida, cv2.MORPH_CLOSE, elemento)
    if preencher:
        saida = preencher_buracos(saida)
    return saida

### 3.8. Etapa F · Contornos, filtros e descritores geométricos

`cv2.findContours` recebe a máscara morfológica com `RETR_EXTERNAL`, o que já evita contar
o miolo da placa como segundo objeto.

| Filtro | Rejeita |
|---|---|
| `área ≥ area_minima` | Ruído residual. Sem ele o ruído entra na contagem |
| `área ≤ area_maxima` | Fachada, toldo e mato seco. A forma não separa da placa de perto, mas a escala separa |
| `0,35 ≤ largura/altura ≤ 2,85` | Faixa, meio-fio e poste |
| `extensão ≥ 0,35` | Contorno rendilhado, normalmente vegetação e sombra |
| `solidez ≥ 0,70` | Forma côncava. Toda placa normativa é convexa |

Os dois limites de área são as pontas do mesmo argumento, e saem da distribuição de áreas
anotadas (Seção 5.2).

**Descritores registrados:** área, perímetro, centroide, circularidade, solidez, extensão,
razão de aspecto, vértices e os sete momentos de Hu, invariantes a translação, escala e
rotação. Nenhum deles decide nada, todos são saída numérica.

**Classificação geométrica.** Os limiares abaixo saíram de medição: cada forma normativa
foi rasterizada em cinco escalas (`r = 12` a `100 px`).

| Forma | Vértices (ε = 2,5% do perímetro) | Circularidade | Extensão | Área / círculo mínimo |
|---|---|---|---|---|
| Triangular (R-2) | 3 | 0,55 | ~0,50 | · |
| Losangular (A-*) | 4 | 0,76 a 0,78 | **~0,50** | · |
| Retangular | 4 | 0,74 | **~1,00** | · |
| Circular | ≥ 5 | 0,86 a 0,89 | ~0,79 | **0,93 a 0,99** |
| Octogonal (R-1) | ≥ 5 | 0,95 | ~0,83 | **0,88 a 0,90** |

Duas consequências:

- **Losango e retângulo separam-se pela extensão**, e não pelo ângulo do `minAreaRect`,
  cuja convenção mudou entre OpenCV 4 e 5.
- **Círculo e octógono só se separam em visada frontal.** Com elongação acima de 1,15 o
  objeto é rotulado `circular`, a classe majoritária, e marcado com `forma_ambigua = True`.
  Boa parte dessa elongação vinha do esticamento da base original, desfeito na Seção 1; o
  que sobra é perspectiva real e tamanho do objeto.


In [ ]:
def classificar_geometria(contorno, extensao: float, epsilon: float = 0.025) -> tuple[str, int, bool]:
    """Classe geometrica: (forma, n_vertices, forma_ambigua). Limiares medidos na celula anterior."""
    perimetro = cv2.arcLength(contorno, True)
    if perimetro <= 0:
        return "indefinido", 0, False

    vertices = len(cv2.approxPolyDP(contorno, epsilon * perimetro, True))

    if vertices == 3:
        return "triangular", vertices, False

    if vertices == 4:
        if extensao >= 0.72:
            return "retangular", vertices, False
        if extensao <= 0.62:
            return "losango", vertices, False
        return "quadrilatero", vertices, True

    if vertices >= 5:
        area = cv2.contourArea(contorno)
        _, raio = cv2.minEnclosingCircle(contorno)
        razao_circulo = area / (math.pi * raio ** 2) if raio > 0 else 0.0

        # Elongacao da elipse ajustada: mede o quanto a visada e obliqua.
        elongacao = 1.0
        if len(contorno) >= 5:
            (_, _), (eixo_maior, eixo_menor), _ = cv2.fitEllipse(contorno)
            if min(eixo_maior, eixo_menor) > 0:
                elongacao = max(eixo_maior, eixo_menor) / min(eixo_maior, eixo_menor)

        if elongacao > 1.15:
            # Sob perspectiva, circulo e octogono sao indistinguiveis por descritor
            # classico: assume-se a classe majoritaria e marca-se a ambiguidade.
            return "circular", vertices, True
        return ("circular" if razao_circulo >= 0.92 else "octogonal"), vertices, False

    return "indefinido", vertices, True


def descrever_contorno(contorno) -> dict:
    """Descritores geometricos de um contorno aceito."""
    area = float(cv2.contourArea(contorno))
    perimetro = float(cv2.arcLength(contorno, True))
    x, y, largura, altura = cv2.boundingRect(contorno)

    momentos = cv2.moments(contorno)
    cx = momentos["m10"] / momentos["m00"] if momentos["m00"] else x + largura / 2.0
    cy = momentos["m01"] / momentos["m00"] if momentos["m00"] else y + altura / 2.0

    # Momentos de Hu: 7 numeros invariantes a translacao, escala e rotacao. Os valores
    # brutos variam por varias ordens de grandeza, entao vao na escala log usual,
    # -sinal(h) * log10(|h|), que e a forma comparavel entre objetos de tamanhos diferentes.
    hu = cv2.HuMoments(momentos).ravel()
    hu_log = [float(-math.copysign(1.0, h) * math.log10(abs(h))) if h else 0.0 for h in hu]

    circularidade = (4.0 * math.pi * area / perimetro ** 2) if perimetro > 0 else 0.0
    casco = cv2.convexHull(contorno)
    area_casco = float(cv2.contourArea(casco))
    solidez = area / area_casco if area_casco > 0 else 0.0
    extensao = area / float(largura * altura) if largura * altura else 0.0
    razao = largura / float(altura) if altura else 0.0

    forma, vertices, ambigua = classificar_geometria(contorno, extensao)
    if forma == "triangular":
        # Apice para baixo (R-2 "De a preferencia") desloca o centroide para cima
        # em relacao ao centro da caixa envolvente.
        forma = "triangular_invertido" if cy < y + altura / 2.0 else "triangular"

    return {
        "area_px": area, "perimetro_px": perimetro,
        "x": int(x), "y": int(y), "w": int(largura), "h": int(altura),
        "cx": float(cx), "cy": float(cy),
        "circularidade": float(circularidade), "solidez": float(solidez),
        "extensao": float(extensao), "razao_aspecto": float(razao),
        "vertices": int(vertices), "forma": forma, "forma_ambigua": bool(ambigua),
        **{f"hu{i}": valor for i, valor in enumerate(hu_log, 1)},
    }


def extrair_contornos(mascara: np.ndarray, area_minima: int = 200,
                      area_maxima: int = 0,
                      razao_aspecto: tuple = (0.35, 2.85),
                      extensao_minima: float = 0.35,
                      solidez_minima: float = 0.70) -> tuple[list, list, dict]:
    """Contornos externos filtrados por area e forma. Devolve (contornos, descritores, stats)."""
    brutos = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    aceitos, descritores = [], []
    motivos = {"area_min": 0, "area_max": 0, "aspecto": 0, "extensao": 0, "solidez": 0}

    for contorno in brutos:
        area = cv2.contourArea(contorno)
        if area < area_minima:
            motivos["area_min"] += 1
            continue
        if area_maxima and area > area_maxima:
            motivos["area_max"] += 1
            continue
        d = descrever_contorno(contorno)
        if not (razao_aspecto[0] <= d["razao_aspecto"] <= razao_aspecto[1]):
            motivos["aspecto"] += 1
            continue
        if d["extensao"] < extensao_minima:
            motivos["extensao"] += 1
            continue
        if d["solidez"] < solidez_minima:
            motivos["solidez"] += 1
            continue
        aceitos.append(contorno)
        descritores.append(d)

    return aceitos, descritores, {
        "contornos_brutos": len(brutos), "aceitos": len(aceitos),
        "descartados": len(brutos) - len(aceitos), "por_motivo": motivos,
    }

### 3.9. Bordas (Sobel e Canny), fora da contagem

O Sobel aproxima o gradiente por convolução; o Canny acrescenta supressão de não-máximos e
histerese, devolvendo borda de um pixel.

**Nenhum dos dois alimenta o `findContours`, e isso é de propósito:** borda fechada de um
pixel tem dois lados, e a contagem dobraria. Os contornos saem sempre da máscara
morfológica preenchida. O Canny fica como evidência visual da estrutura da cena.


In [ ]:
def bordas_sobel(rgb: np.ndarray, k: int = 3) -> np.ndarray:
    """Magnitude do gradiente por Sobel, normalizada para uint8."""
    cinza = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(cinza, cv2.CV_32F, 1, 0, ksize=impar(k))
    gy = cv2.Sobel(cinza, cv2.CV_32F, 0, 1, ksize=impar(k))
    magnitude = cv2.magnitude(gx, gy)
    return cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


def bordas_canny(rgb: np.ndarray, sigma: float = 0.33) -> np.ndarray:
    """Canny com histerese ancorada na mediana da imagem (sem constantes magicas)."""
    cinza = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    mediana = float(np.median(cinza))
    inferior = int(max(0, (1.0 - sigma) * mediana))
    superior = int(min(255, (1.0 + sigma) * mediana))
    return cv2.Canny(cinza, inferior, superior)

---
## 4. Orquestrador e visualização

`executar_pipeline` encadeia as funções da Seção 3 e devolve todos os estágios
intermediários, o que permite montar os painéis de antes e depois sem reprocessar a imagem.


In [ ]:
def executar_pipeline(entrada, params: Parametros) -> dict:
    """Pipeline completo sobre uma imagem: devolve etapas, contornos, descritores e limiar."""
    rgb = carregar_imagem(entrada) if isinstance(entrada, (str, Path)) else entrada
    etapas = {}

    imagem, escala = redimensionar(rgb, params.largura_trabalho)
    etapas["1_original"] = imagem

    etapas["2_iluminacao"] = (corrigir_iluminacao_clahe(imagem, params.clahe_clip, params.clahe_grade)
                              if params.usar_clahe else imagem)
    etapas["3_suavizacao"] = suavizar(etapas["2_iluminacao"], params.suavizacao, params.suavizacao_k)
    etapas["4_evidencia"] = mapa_evidencia_cromatica(etapas["3_suavizacao"], params.faixas)

    mascara, limiar = limiarizar(etapas["4_evidencia"], params.metodo_limiar,
                                 params.limiar_global, params.adapt_bloco, params.adapt_c)
    etapas["5_binaria"] = mascara
    etapas["6_morfologia"] = limpar_mascara(mascara, params.k_abertura, params.k_fechamento,
                                            params.preencher_buracos)

    contornos, objetos, stats = extrair_contornos(
        etapas["6_morfologia"], params.area_minima, params.area_maxima,
        params.razao_aspecto, params.extensao_minima, params.solidez_minima)

    return {"etapas": etapas, "contornos": contornos, "objetos": objetos,
            "limiar": limiar, "escala": escala, "estatisticas": stats,
            "dimensoes_trabalho": imagem.shape[:2]}


def recortar_regioes(resultado: dict, margem: float = 0.12) -> list[np.ndarray]:
    """Recorta a ROI normalizada de cada objeto, que e a entrada da IA na N2."""
    imagem = resultado["etapas"]["1_original"]
    altura, largura = imagem.shape[:2]
    recortes = []
    for obj in resultado["objetos"]:
        mx, my = int(obj["w"] * margem), int(obj["h"] * margem)
        x0, y0 = max(0, obj["x"] - mx), max(0, obj["y"] - my)
        x1, y1 = min(largura, obj["x"] + obj["w"] + mx), min(altura, obj["y"] + obj["h"] + my)
        if x1 > x0 and y1 > y0:
            recortes.append(imagem[y0:y1, x0:x1])
    return recortes

In [ ]:
CORES_FORMA = {
    "circular": (231, 76, 60), "octogonal": (192, 57, 43),
    "triangular": (243, 156, 18), "triangular_invertido": (211, 84, 0),
    "losango": (241, 196, 15), "retangular": (41, 128, 185),
    "quadrilatero": (127, 140, 141), "indefinido": (149, 165, 166),
}


def desenhar_deteccoes(resultado: dict, espessura: int = 2) -> np.ndarray:
    """Sobrepoe contorno, caixa, centroide e rotulo de forma sobre a imagem original."""
    tela = resultado["etapas"]["1_original"].copy()
    for i, (contorno, obj) in enumerate(zip(resultado["contornos"], resultado["objetos"]), 1):
        cor = CORES_FORMA.get(obj["forma"], (149, 165, 166))
        cv2.drawContours(tela, [contorno], -1, cor, espessura)
        cv2.rectangle(tela, (obj["x"], obj["y"]),
                      (obj["x"] + obj["w"], obj["y"] + obj["h"]), cor, 1)
        cv2.circle(tela, (int(obj["cx"]), int(obj["cy"])), 3, (255, 255, 255), -1)
        cv2.circle(tela, (int(obj["cx"]), int(obj["cy"])), 3, cor, 1)
        rotulo = f"{i} {obj['forma']}{'?' if obj['forma_ambigua'] else ''}"
        y_texto = max(12, obj["y"] - 6)
        cv2.putText(tela, rotulo, (obj["x"], y_texto),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(tela, rotulo, (obj["x"], y_texto),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, cor, 1, cv2.LINE_AA)
    return tela


TITULOS_ETAPAS = {
    "1_original": "1. Entrada redimensionada",
    "2_iluminacao": "2. Correção de iluminação (CLAHE em L*)",
    "3_suavizacao": "3. Suavização (Gaussiano)",
    "4_evidencia": "4. Mapa de evidência cromática (HSV)",
    "5_binaria": "5. Limiarização",
    "6_morfologia": "6. Morfologia + preenchimento",
}


def painel_pipeline(resultado: dict, titulo: str = "", salvar: Path | None = None):
    """Grade 2x4 com as seis etapas, o Canny (fora da contagem) e o resultado final."""
    etapas = resultado["etapas"]
    canny = bordas_canny(etapas["3_suavizacao"])
    quadros = [(TITULOS_ETAPAS[k], etapas[k]) for k in TITULOS_ETAPAS]
    quadros.append(("7. Canny (evidência visual,\nnão alimenta findContours)", canny))
    n_obj = len(resultado["objetos"])
    quadros.append((f"8. Saída: {n_obj} objeto(s) detectado(s)", desenhar_deteccoes(resultado)))

    fig, eixos = plt.subplots(2, 4, figsize=(16, 7.2))
    for eixo, (nome, imagem) in zip(eixos.ravel(), quadros):
        if imagem.ndim == 2:
            eixo.imshow(imagem, cmap="magma" if "evidência" in nome else "gray", vmin=0, vmax=255)
        else:
            eixo.imshow(imagem)
        eixo.set_title(nome, fontsize=9)
        eixo.axis("off")
    limiar = resultado["limiar"]
    texto_limiar = "local" if np.isnan(limiar) else f"{limiar:.0f}"
    fig.suptitle(f"{titulo}   ·   limiar aplicado: {texto_limiar}   ·   "
                 f"contornos brutos: {resultado['estatisticas']['contornos_brutos']} → "
                 f"aceitos: {resultado['estatisticas']['aceitos']}",
                 fontsize=11, y=1.0)
    fig.tight_layout()
    if salvar:
        salvar_figura(fig, salvar)
    return fig

---
## 5. Calibração dos parâmetros

Limiar, kernel e área mínima são obtidos por medição, em três partes:

- **5.0. Faixas de cor:** confrontadas com as anotações, pixel a pixel.
- **5.1. Kernels:** consequência geométrica da escala do objeto, medida em 2.1.
- **5.2. Portas de saturação, área e método:** governam o compromisso entre precisão e
  recall, então são escolhidos por métrica, com critério declarado antes.

### Protocolo anti-viés

As decisões usam a amostra de ajuste; as métricas da Seção 8 vêm de uma amostra de
validação que não participou de nenhuma escolha.

**A divisão é por trecho de gravação, não por imagem.** A base são quadros de câmera
veicular, um a cada 3 segundos em média. Sorteando imagem por imagem, a mesma placa vista
um segundo depois cairia nos dois lados: numa versão anterior, 55% das imagens de validação
tinham um quadro de ajuste a menos de 30 s. Agora um trecho novo começa depois de 30 s sem
captura (a 80 km/h, uns 650 m) e vai inteiro para um lado só. A célula imprime a menor
distância no tempo entre os dois lados, como conferência.


In [ ]:
# Quadros da mesma gravacao tirados a poucos segundos um do outro mostram a mesma placa. Se
# um cai no ajuste e o vizinho na validacao, a validacao deixa de ser independente. Por isso
# a divisao e feita por trecho: um trecho novo comeca depois de LIMITE_TRECHO_S segundos sem
# captura, e todos os quadros de um trecho vao pro mesmo lado.
LIMITE_TRECHO_S = 30
PADRAO_INSTANTE = r"captura_(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})"


def atribuir_trechos(inventario: pd.DataFrame, limite_s: float) -> tuple[pd.Series, pd.Series]:
    """Trecho de gravacao e instante de cada imagem; sem data no nome, vira trecho sozinha."""
    instantes = pd.to_datetime(inventario["arquivo"].str.extract(PADRAO_INSTANTE)[0],
                               format="%Y-%m-%d_%H-%M-%S", errors="coerce")
    trechos = pd.Series(0, index=inventario.index)
    atual, anterior = 0, None
    for idx in instantes.sort_values(na_position="last").index:
        instante = instantes[idx]
        if pd.isna(instante) or anterior is None or (instante - anterior).total_seconds() > limite_s:
            atual += 1
        trechos[idx] = atual
        anterior = None if pd.isna(instante) else instante
    return trechos, instantes


INVENTARIO["trecho"], INVENTARIO["instante"] = atribuir_trechos(INVENTARIO, LIMITE_TRECHO_S)
anotadas = INVENTARIO[INVENTARIO["n_objetos"] > 0]

# Os trechos sao embaralhados com a semente e distribuidos alternando pro lado que tem menos
# imagens anotadas ate o momento, pra que os dois lados fiquem com tamanhos parecidos.
_por_trecho = anotadas.groupby("trecho").size().sample(frac=1, random_state=SEMENTE)
LADO_DO_TRECHO, _total = {}, {"ajuste": 0, "validacao": 0}
for trecho, quantidade in _por_trecho.items():
    lado = "ajuste" if _total["ajuste"] <= _total["validacao"] else "validacao"
    LADO_DO_TRECHO[trecho] = lado
    _total[lado] += int(quantidade)

POOL_AJUSTE = anotadas[anotadas["trecho"].map(LADO_DO_TRECHO) == "ajuste"]
POOL_VALIDACAO = anotadas[anotadas["trecho"].map(LADO_DO_TRECHO) == "validacao"]

AMOSTRA_AJUSTE = [Path(p) for p in POOL_AJUSTE.sample(
    min(N_AMOSTRA_VALIDACAO, len(POOL_AJUSTE)), random_state=SEMENTE)["caminho"]]       # mede e escolhe
AMOSTRA_VALIDACAO = [Path(p) for p in POOL_VALIDACAO.sample(
    min(N_AMOSTRA_VALIDACAO, len(POOL_VALIDACAO)), random_state=SEMENTE)["caminho"]]   # so mede

assert not (set(AMOSTRA_AJUSTE) & set(AMOSTRA_VALIDACAO)), "as amostras precisam ser disjuntas"
assert not (set(POOL_AJUSTE["trecho"]) & set(POOL_VALIDACAO["trecho"])), "trecho nos dois lados"

# Conferencia: menor distancia no tempo entre um quadro de validacao e um de ajuste.
_t_ajuste = np.sort(POOL_AJUSTE["instante"].dropna().values.astype("datetime64[s]").astype(np.int64))
_t_valid = POOL_VALIDACAO["instante"].dropna().values.astype("datetime64[s]").astype(np.int64)
if len(_t_ajuste) and len(_t_valid):
    _pos = np.clip(np.searchsorted(_t_ajuste, _t_valid), 1, len(_t_ajuste) - 1)
    _dist = np.minimum(np.abs(_t_valid - _t_ajuste[_pos - 1]), np.abs(_t_valid - _t_ajuste[_pos]))
    MENOR_DISTANCIA_S = int(_dist.min())
else:
    MENOR_DISTANCIA_S = None

DIVISAO = {
    "criterio": f"por trecho de gravacao (novo trecho apos {LIMITE_TRECHO_S} s sem captura)",
    "trechos_com_anotacao": int(anotadas["trecho"].nunique()),
    "trechos_ajuste": int(POOL_AJUSTE["trecho"].nunique()),
    "trechos_validacao": int(POOL_VALIDACAO["trecho"].nunique()),
    "imagens_anotadas_ajuste": int(len(POOL_AJUSTE)),
    "imagens_anotadas_validacao": int(len(POOL_VALIDACAO)),
    "menor_distancia_ajuste_validacao_s": MENOR_DISTANCIA_S,
}
print(f"Trechos de gravacao com anotacao: {DIVISAO['trechos_com_anotacao']} "
      f"(novo trecho apos {LIMITE_TRECHO_S} s sem captura)")
print(f"  lado de ajuste    : {DIVISAO['trechos_ajuste']} trechos, {DIVISAO['imagens_anotadas_ajuste']} imagens anotadas")
print(f"  lado de validacao : {DIVISAO['trechos_validacao']} trechos, {DIVISAO['imagens_anotadas_validacao']} imagens anotadas")
print(f"  menor distancia no tempo entre um quadro de ajuste e um de validacao: {MENOR_DISTANCIA_S} s")
print(f"\nAmostra de ajuste    : {len(AMOSTRA_AJUSTE)} imagens")
print(f"Amostra de validação : {len(AMOSTRA_VALIDACAO)} imagens (trechos disjuntos, semente {SEMENTE})")


### 5.0. O que as anotações dizem sobre a cor

As âncoras vêm da norma. O que a norma não diz é quais cores existem nesta base, onde elas
caem na escala `H` e o quanto se confundem com o fundo. Três passos:

1. **Perfil de cada classe:** recorta o núcleo central (60%) das caixas, pega os pixels
   saturados e calcula a matiz dominante por média circular. Classe com pictograma preto
   sobre branco não produz matiz, e isso também é resultado.
2. **Formação das faixas:** cada classe vai para a âncora mais próxima em distância
   circular. A âncora só vira faixa com objetos anotados suficientes, e o centro e a largura
   saem da distribuição observada.
3. **Placa contra fundo:** para cada faixa e cada candidato a `s_min`, mede que fração de
   pixel de placa e de fundo sobrevive à porta. Porta boa corta muito fundo e pouca placa.


In [ ]:
MIN_OBJETOS_POR_FAIXA = 25    # abaixo disso, a cor nao justifica uma faixa no pipeline
MIN_OBJETOS_POR_CLASSE = 5    # abaixo disso, o perfil de cor da classe e ruido
S_REFERENCIA = 90             # saturacao acima da qual a matiz do pixel e considerada confiavel


def _media_circular_matiz(h: np.ndarray, periodo: float = 180.0) -> float:
    """Media de matizes num circulo: evita que 179 e 1 tenham media 90."""
    angulos = h.astype(np.float64) * (2 * np.pi / periodo)
    media = np.arctan2(np.sin(angulos).mean(), np.cos(angulos).mean())
    return float((media % (2 * np.pi)) * periodo / (2 * np.pi))


def perfilar_classes(caminhos: list[Path], params: Parametros, nucleo: float = 0.6,
                     max_pixels: int = 4000) -> pd.DataFrame:
    """Matiz e saturacao dominantes de cada classe anotada, medidas no nucleo das caixas."""
    coleta: dict[int, dict[str, list]] = {}
    for caminho in caminhos:
        rotulos = ler_rotulos(caminho)
        if not rotulos:
            continue
        imagem, _ = redimensionar(carregar_imagem(caminho), params.largura_trabalho)
        if params.usar_clahe:
            imagem = corrigir_iluminacao_clahe(imagem, params.clahe_clip, params.clahe_grade)
        canal_h, canal_s, _ = cv2.split(cv2.cvtColor(imagem, cv2.COLOR_RGB2HSV))
        altura, largura = canal_h.shape
        for cls, cx, cy, w, h in rotulos:
            x0, x1 = int((cx - w / 2) * largura), int((cx + w / 2) * largura)
            y0, y1 = int((cy - h / 2) * altura), int((cy + h / 2) * altura)
            mx, my = int((x1 - x0) * (1 - nucleo) / 2), int((y1 - y0) * (1 - nucleo) / 2)
            x0, x1, y0, y1 = max(0, x0 + mx), x1 - mx, max(0, y0 + my), y1 - my
            dados = coleta.setdefault(cls, {"n": 0, "h": [], "s": []})
            dados["n"] += 1
            if x1 <= x0 or y1 <= y0:
                continue
            recorte_s = canal_s[y0:y1, x0:x1]
            saturado = recorte_s >= S_REFERENCIA
            if saturado.any():
                dados["h"].extend(canal_h[y0:y1, x0:x1][saturado].tolist()[:max_pixels])
                dados["s"].extend(recorte_s[saturado].tolist()[:max_pixels])

    linhas = []
    for cls, dados in coleta.items():
        matizes = np.array(dados["h"])
        linhas.append({
            "classe": CLASSES[cls] if cls < len(CLASSES) else str(cls),
            "indice": cls,
            "objetos": dados["n"],
            "pixels_saturados": int(matizes.size),
            "h_dominante": round(_media_circular_matiz(matizes), 1) if matizes.size else np.nan,
            "s_mediana": round(float(np.median(dados["s"])), 0) if dados["s"] else np.nan,
        })
    return pd.DataFrame(linhas).sort_values("objetos", ascending=False).reset_index(drop=True)


def ancora_mais_proxima(matiz: float, ancoras: list) -> str:
    """Nome da ancora normativa mais proxima de `matiz`, em distancia circular."""
    valor = np.array([[matiz]], np.float32)
    return min(ancoras, key=lambda a: float(distancia_circular_matiz(valor, a["h_centro"])[0, 0]))["nome"]


def atribuir_classes(perfil: pd.DataFrame, ancoras: list) -> pd.DataFrame:
    """Anexa a `perfil` a coluna `faixa`: a cor normativa mais proxima da matiz medida."""
    perfil = perfil.copy()
    perfil["faixa"] = [
        ancora_mais_proxima(h, ancoras)
        if np.isfinite(h) and n >= MIN_OBJETOS_POR_CLASSE else "sem matiz"
        for h, n in zip(perfil["h_dominante"], perfil["objetos"])]
    return perfil


def formar_faixas(perfil: pd.DataFrame, ancoras: list) -> tuple[list, pd.DataFrame]:
    """Atribui cada classe a ancora normativa mais proxima e recalcula centro e largura."""
    perfil = atribuir_classes(perfil, ancoras)
    faixas, resumo = [], []
    for ancora in ancoras:
        classes = perfil[perfil["faixa"] == ancora["nome"]]
        objetos = int(classes["objetos"].sum())
        matizes = np.repeat(classes["h_dominante"].to_numpy(dtype=float),
                            classes["objetos"].to_numpy(dtype=int)) if len(classes) else np.array([])
        if objetos >= MIN_OBJETOS_POR_FAIXA and matizes.size:
            centro = _media_circular_matiz(matizes)
            desvio = np.abs(distancia_circular_matiz(matizes.astype(np.float32)[None, :], centro))
            sigma = float(np.clip(np.percentile(desvio, 90), 6.0, 20.0))
            faixas.append({**ancora, "h_centro": round(centro, 1), "h_sigma": round(sigma, 1)})
        resumo.append({"faixa": ancora["nome"], "ancora_h": ancora["h_centro"],
                       "classes": len(classes), "objetos": objetos,
                       "h_medido": round(_media_circular_matiz(matizes), 1) if matizes.size else np.nan,
                       "vira_faixa": objetos >= MIN_OBJETOS_POR_FAIXA and bool(matizes.size)})
    return faixas, pd.DataFrame(resumo)


PERFIL_CLASSES = perfilar_classes(AMOSTRA_AJUSTE, Parametros())
FAIXAS_MEDIDAS, RESUMO_FAIXAS = formar_faixas(PERFIL_CLASSES, FAIXAS_CONTRAN)
FAIXA_DA_CLASSE = dict(zip(PERFIL_CLASSES["indice"],
                           atribuir_classes(PERFIL_CLASSES, FAIXAS_CONTRAN)["faixa"]))

print(f"Classes anotadas na amostra de ajuste: {len(PERFIL_CLASSES)} "
      f"({int(PERFIL_CLASSES['objetos'].sum())} objetos)\n")
print("Perfil das classes mais frequentes (matiz dominante no nucleo das caixas):")
print(PERFIL_CLASSES.head(12)[["classe", "objetos", "h_dominante", "s_mediana"]].to_string(index=False))
print("\nFormacao das faixas a partir das ancoras normativas:")
print(RESUMO_FAIXAS.to_string(index=False))
print("\nFaixas adotadas pelo pipeline:")
for faixa in FAIXAS_MEDIDAS:
    print(f"  {faixa['nome']:9s} H {faixa['h_centro']:5.1f} +/- {faixa['h_sigma']:4.1f}  "
          f"(ancora era H {next(a['h_centro'] for a in FAIXAS_CONTRAN if a['nome'] == faixa['nome']):5.1f})")
for ancora in FAIXAS_CONTRAN:
    if not any(f["nome"] == ancora["nome"] for f in FAIXAS_MEDIDAS):
        print(f"  {ancora['nome']:9s} FORA: menos de {MIN_OBJETOS_POR_FAIXA} objetos anotados nesta cor")

In [ ]:
def medir_portas(caminhos: list[Path], params: Parametros, nucleo: float = 0.6,
                 max_fundo_por_imagem: int = 4000) -> dict:
    """Saturacao dos pixels de placa e de fundo, dentro de cada faixa de matiz."""
    s_placa = {f["nome"]: [] for f in params.faixas}
    s_fundo = {f["nome"]: [] for f in params.faixas}
    gerador = np.random.default_rng(SEMENTE)

    for caminho in caminhos:
        imagem, _ = redimensionar(carregar_imagem(caminho), params.largura_trabalho)
        if params.usar_clahe:
            imagem = corrigir_iluminacao_clahe(imagem, params.clahe_clip, params.clahe_grade)
        imagem = suavizar(imagem, params.suavizacao, params.suavizacao_k)
        canal_h, canal_s, _ = cv2.split(cv2.cvtColor(imagem, cv2.COLOR_RGB2HSV))
        altura, largura = canal_h.shape
        caixas = np.zeros((altura, largura), np.uint8)
        nucleos = np.zeros((altura, largura), bool)
        for _cls, cx, cy, w, h in ler_rotulos(caminho):
            x0, x1 = int((cx - w / 2) * largura), int((cx + w / 2) * largura)
            y0, y1 = int((cy - h / 2) * altura), int((cy + h / 2) * altura)
            cv2.rectangle(caixas, (x0 - 8, y0 - 8), (x1 + 8, y1 + 8), 255, -1)
            mx, my = int((x1 - x0) * (1 - nucleo) / 2), int((y1 - y0) * (1 - nucleo) / 2)
            if x1 - mx > x0 + mx and y1 - my > y0 + my:
                nucleos[y0 + my:y1 - my, x0 + mx:x1 - mx] = True
        fundo = caixas == 0
        for faixa in params.faixas:
            proximo = distancia_circular_matiz(canal_h, faixa["h_centro"]) <= 1.5 * faixa["h_sigma"]
            s_placa[faixa["nome"]].append(canal_s[nucleos & proximo])
            indices = np.flatnonzero(fundo & proximo)
            if indices.size > max_fundo_por_imagem:
                indices = gerador.choice(indices, max_fundo_por_imagem, replace=False)
            s_fundo[faixa["nome"]].append(canal_s.ravel()[indices])

    return {"s_placa": {k: np.concatenate(v) if v else np.array([]) for k, v in s_placa.items()},
            "s_fundo": {k: np.concatenate(v) if v else np.array([]) for k, v in s_fundo.items()}}


PARAMS_CROMA = Parametros(faixas=FAIXAS_MEDIDAS)
CROMATICIDADE = medir_portas(AMOSTRA_AJUSTE, PARAMS_CROMA)
CANDIDATOS_S = [80, 100, 120, 140, 160, 180]

linhas = []
for faixa in FAIXAS_MEDIDAS:
    placa, fundo = CROMATICIDADE["s_placa"][faixa["nome"]], CROMATICIDADE["s_fundo"][faixa["nome"]]
    for s in CANDIDATOS_S:
        linhas.append({"faixa": faixa["nome"], "s_min": s,
                       "placa mantida": round(float((placa >= s).mean()), 2) if placa.size else np.nan,
                       "fundo mantido": round(float((fundo >= s).mean()), 2) if fundo.size else np.nan})
TABELA_PORTAS = pd.DataFrame(linhas).pivot(index="s_min", columns="faixa",
                                           values=["placa mantida", "fundo mantido"])
print("Fracao de pixels que sobrevive a cada porta de saturacao (matiz dentro da faixa):")
print(TABELA_PORTAS.to_string())
print("\nLeitura: a coluna 'placa mantida' e o que a porta preserva do alvo; 'fundo mantido' e o")
print("que ela deixa entrar de ceu, asfalto, vegetacao e fachada. O estagio 0 da Secao 5.2")
print("escolhe o ponto por F1, e pode descartar uma faixa inteira.")

In [ ]:
CORES_FAIXA = {"vermelho": "#c0392b", "amarelo": "#e0a800", "verde": "#2e8b57", "azul": "#2d6a9f"}

n_faixas = len(FAIXAS_MEDIDAS)
fig, eixos = plt.subplots(1, n_faixas + 1, figsize=(3.8 * (n_faixas + 1), 3.6))
for eixo, faixa in zip(eixos, FAIXAS_MEDIDAS):
    nome = faixa["nome"]
    placa, fundo = CROMATICIDADE["s_placa"][nome], CROMATICIDADE["s_fundo"][nome]
    if fundo.size:
        eixo.hist(fundo, bins=32, range=(0, 255), density=True, alpha=0.6, color="#7f8c8d", label="fundo")
    if placa.size:
        eixo.hist(placa, bins=32, range=(0, 255), density=True, alpha=0.6,
                  color=CORES_FAIXA.get(nome, "#555555"), label="placa (núcleo da caixa)")
    eixo.axvline(faixa["s_min"], color="#1b262c", ls="--", lw=1.5, label=f"s_min inicial = {faixa['s_min']}")
    eixo.set_title(f"Saturação na faixa {nome}\nH {faixa['h_centro']:.0f} ± {faixa['h_sigma']:.0f}")
    eixo.set_xlabel("S"); eixo.legend(fontsize=7.5)

# Ultimo painel: mostra onde cada classe do dataset caiu na escala de matiz.
eixo = eixos[-1]
com_matiz = PERFIL_CLASSES.dropna(subset=["h_dominante"])
com_matiz = com_matiz[com_matiz["objetos"] >= MIN_OBJETOS_POR_CLASSE]
for faixa in FAIXAS_MEDIDAS:
    inicio = faixa["h_centro"] - 1.5 * faixa["h_sigma"]
    fim = faixa["h_centro"] + 1.5 * faixa["h_sigma"]
    trechos = [(max(0.0, inicio), min(180.0, fim))]
    if inicio < 0:                      # a faixa vermelha da a volta no circulo de matiz
        trechos.append((180.0 + inicio, 180.0))
    if fim > 180:
        trechos.append((0.0, fim - 180.0))
    for x0, x1 in trechos:
        eixo.axvspan(x0, x1, color=CORES_FAIXA.get(faixa["nome"], "#999999"), alpha=0.15)
eixo.scatter(com_matiz["h_dominante"], com_matiz["objetos"], s=22, color="#1b262c", zorder=3)
for _, linha in com_matiz.head(14).iterrows():
    eixo.annotate(linha["classe"], (linha["h_dominante"], linha["objetos"]),
                  fontsize=6.5, xytext=(3, 3), textcoords="offset points")
eixo.set_yscale("log"); eixo.set_xlim(0, 180)
eixo.set_title("Matiz dominante de cada classe anotada")
eixo.set_xlabel("H (OpenCV, de 0 a 179)"); eixo.set_ylabel("objetos na amostra (log)")

fig.suptitle("Calibração cromática nas anotações da amostra de ajuste", y=1.03)
fig.tight_layout()
fig.savefig(DIR_FIGURAS / "00b_calibracao_cromatica.png", bbox_inches="tight")
plt.show()

In [ ]:
def mosaico_recortes(caminhos: list[Path], por_faixa: int = 6, lado: int = 96,
                     salvar: Path | None = None):
    """Recorta exemplos anotados de cada faixa de cor, para inspecao visual direta."""
    nomes = [f["nome"] for f in FAIXAS_MEDIDAS]
    recortes = {nome: [] for nome in nomes}
    for caminho in caminhos:
        rotulos = ler_rotulos(caminho)
        pendentes = [r for r in rotulos if FAIXA_DA_CLASSE.get(r[0]) in nomes
                     and len(recortes[FAIXA_DA_CLASSE[r[0]]]) < por_faixa]
        if not pendentes:
            continue
        rgb = carregar_imagem(caminho)
        altura, largura = rgb.shape[:2]
        for cls, cx, cy, w, h in pendentes:
            x0, x1 = max(0, int((cx - w / 2) * largura)), int((cx + w / 2) * largura)
            y0, y1 = max(0, int((cy - h / 2) * altura)), int((cy + h / 2) * altura)
            if x1 - x0 < 10 or y1 - y0 < 10:
                continue
            recorte = cv2.resize(rgb[y0:y1, x0:x1], (lado, lado), interpolation=cv2.INTER_CUBIC)
            rotulo = CLASSES[cls] if cls < len(CLASSES) else str(cls)
            cv2.putText(recorte, rotulo[:9], (3, 13), cv2.FONT_HERSHEY_SIMPLEX, 0.42,
                        (255, 255, 0), 1, cv2.LINE_AA)
            recortes[FAIXA_DA_CLASSE[cls]].append(recorte)
        if all(len(v) >= por_faixa for v in recortes.values()):
            break

    fig, eixos = plt.subplots(len(nomes), por_faixa,
                              figsize=(1.55 * por_faixa, 1.75 * len(nomes)))
    eixos = np.atleast_2d(eixos)
    for linha, nome in enumerate(nomes):
        for coluna in range(por_faixa):
            eixo = eixos[linha, coluna]
            eixo.axis("off")
            if coluna < len(recortes[nome]):
                eixo.imshow(recortes[nome][coluna])
        eixos[linha, 0].set_ylabel(nome)
        eixos[linha, 0].axis("on"); eixos[linha, 0].set_xticks([]); eixos[linha, 0].set_yticks([])
    fig.suptitle("Recortes anotados por faixa de cor medida (rótulo = código CONTRAN)", y=1.0)
    fig.tight_layout()
    if salvar:
        salvar_figura(fig, salvar)
    return fig


mosaico_recortes(AMOSTRA_AJUSTE, salvar=DIR_FIGURAS / "00c_recortes_por_faixa.jpg")
plt.show()

### 5.1. Kernels derivados da escala do objeto

| Parâmetro | Regra | Por quê |
|---|---|---|
| `suavizacao_k` | `ímpar(0,10 × p10 do lado equivalente)` | O borrão precisa ser mais estreito que a orla da menor placa detectável |
| `k_abertura` | `ímpar(0,10 × p10 do lado equivalente)` | Apaga ruído sem apagar a menor placa, por isso ancora no percentil 10 |
| `k_fechamento` | `ímpar(0,25 × mediana do lado equivalente)` | Precisa vencer a espessura do miolo branco de uma placa típica |

O lado equivalente é medido nas caixas do lado de ajuste. Sem anotação, a regra cai para
valores padrão seguros.


In [ ]:
def area_minima_para_quantil(escala: pd.DataFrame, quantil: float,
                             fator_preenchimento: float = 0.45,
                             minimo: int = 30) -> int:
    """Area minima pelo quantil inferior das placas anotadas.

    O fator 0.45 converte area da caixa em area da figura inscrita (losango preenche ~0.50).
    """
    if escala is None or escala.empty:
        return 200
    return int(max(minimo, round(float(escala["area_px"].quantile(quantil)) * fator_preenchimento)))


def area_maxima_para_quantil(escala: pd.DataFrame, quantil: float) -> int:
    """Teto de area pelo quantil superior das placas (`1.0` = sem teto).

    Rejeita ceu, fachada e vegetacao, que tem a cor da placa mas sao muito maiores.
    """
    if escala is None or escala.empty or quantil >= 1.0:
        return 0
    return int(round(float(escala["area_px"].quantile(quantil))))


def calibrar_kernels(escala: pd.DataFrame, base: Parametros | None = None) -> tuple[Parametros, dict]:
    """Deriva os kernels morfologicos da distribuicao de tamanho das placas anotadas."""
    params = base or Parametros()
    if escala is None or escala.empty:
        return params, {"origem": "padrao (dataset sem anotacoes)"}

    lado_p10 = float(escala["lado_equivalente_px"].quantile(0.10))
    lado_mediano = float(escala["lado_equivalente_px"].median())

    params.suavizacao_k = impar(0.10 * lado_p10, minimo=3)
    params.k_abertura = impar(0.10 * lado_p10, minimo=3)
    params.k_fechamento = impar(0.25 * lado_mediano, minimo=3)

    memoria = {
        "origem": f"{len(escala)} caixas anotadas",
        "lado_equivalente_p10_px": round(lado_p10, 1),
        "lado_equivalente_mediano_px": round(lado_mediano, 1),
        "regra_suavizacao_k": f"impar(0.10 x {lado_p10:.1f}) = {params.suavizacao_k}",
        "regra_k_abertura": f"impar(0.10 x {lado_p10:.1f}) = {params.k_abertura}",
        "regra_k_fechamento": f"impar(0.25 x {lado_mediano:.1f}) = {params.k_fechamento}",
    }
    return params, memoria


# A escala da Secao 2.1 descreve a base inteira. Pra calibrar, so vale o lado de ajuste:
# kernels e areas tambem sao parametros, e nao podem ver as anotacoes da validacao.
ESCALA_AJUSTE = estatisticas_de_escala(POOL_AJUSTE, N_AMOSTRA_CALIBRACAO, LARGURA_TRABALHO)
PARAMS, MEMORIA_CALIBRACAO = calibrar_kernels(ESCALA_AJUSTE, Parametros(faixas=FAIXAS_MEDIDAS))
MEMORIA_CALIBRACAO["origem"] += " do lado de ajuste"

print("KERNELS DERIVADOS DA ESCALA DO OBJETO")
print("-" * 58)
for chave, valor in MEMORIA_CALIBRACAO.items():
    print(f"{chave:<32} {valor}")
print("-" * 58)

### 5.2. Portas de saturação, área e método, escolhidos por métrica

Cada objeto detectado é casado com as caixas anotadas por `IoU ≥ 0,30`. O limiar é frouxo
de propósito: o objetivo aqui é localizar para recortar, não delimitar com precisão de
detector treinado. Do casamento saem precisão, recall e F1.


In [ ]:
def iou(caixa_a: tuple, caixa_b: tuple) -> float:
    """Intersecao sobre uniao de duas caixas no formato (x, y, w, h)."""
    ax, ay, aw, ah = caixa_a
    bx, by, bw, bh = caixa_b
    x0, y0 = max(ax, bx), max(ay, by)
    x1, y1 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    intersecao = max(0, x1 - x0) * max(0, y1 - y0)
    uniao = aw * ah + bw * bh - intersecao
    return intersecao / uniao if uniao > 0 else 0.0


def caixas_anotadas(img: Path, largura: int, altura: int) -> list[tuple]:
    """Converte as anotacoes YOLO normalizadas para pixels na resolucao de trabalho."""
    caixas = []
    for _, cx, cy, w, h in ler_rotulos(img):
        caixas.append((int((cx - w / 2) * largura), int((cy - h / 2) * altura),
                       int(w * largura), int(h * altura)))
    return caixas


def casar_deteccoes(detectadas: list, anotadas: list, limiar_iou: float = 0.30) -> tuple[int, int, int]:
    """Casamento guloso por IoU decrescente. Devolve (verdadeiros_pos, falsos_pos, falsos_neg)."""
    pares = sorted(((iou(d, a), i, j) for i, d in enumerate(detectadas)
                    for j, a in enumerate(anotadas)), reverse=True)
    usados_det, usados_anot = set(), set()
    verdadeiros = 0
    for valor, i, j in pares:
        if valor < limiar_iou:
            break
        if i in usados_det or j in usados_anot:
            continue
        usados_det.add(i)
        usados_anot.add(j)
        verdadeiros += 1
    return verdadeiros, len(detectadas) - verdadeiros, len(anotadas) - verdadeiros


def avaliar_configuracao(caminhos: list[Path], params: Parametros,
                         limiar_iou: float = 0.30) -> dict:
    """Roda o pipeline em varias imagens e agrega precisao, recall e F1."""
    vp = fp = fn = 0
    limiares, tempos = [], []
    for caminho in caminhos:
        inicio = time.perf_counter()
        resultado = executar_pipeline(caminho, params)
        tempos.append((time.perf_counter() - inicio) * 1000)
        altura, largura = resultado["dimensoes_trabalho"]
        detectadas = [(o["x"], o["y"], o["w"], o["h"]) for o in resultado["objetos"]]
        a, b, c = casar_deteccoes(detectadas, caixas_anotadas(caminho, largura, altura), limiar_iou)
        vp, fp, fn = vp + a, fp + b, fn + c
        if not np.isnan(resultado["limiar"]):
            limiares.append(resultado["limiar"])

    precisao = vp / (vp + fp) if vp + fp else 0.0
    recall = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * precisao * recall / (precisao + recall) if precisao + recall else 0.0
    return {"precisao": round(precisao, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "vp": vp, "fp": fp, "fn": fn,
            "limiar_medio": round(float(np.mean(limiares)), 1) if limiares else float("nan"),
            "limiar_desvio": round(float(np.std(limiares)), 1) if limiares else float("nan"),
            "ms_por_imagem": round(float(np.mean(tempos)), 1)}


def com_parametros(base: Parametros, **alteracoes) -> Parametros:
    """Copia `base` com as alteracoes pedidas, pra nao mutar a configuracao atual."""
    novo = Parametros(**{**base.para_dict(), **alteracoes})
    novo.razao_aspecto = tuple(novo.razao_aspecto)
    return novo

#### A busca em grade

Cada candidato a área mínima é expresso como "descartar o quantil inferior `q` das placas
anotadas", e não como número solto de pixel, para o parâmetro ter significado.

São quatro estágios de busca por coordenadas, todos decidindo na amostra de ajuste:

0. **Portas de saturação** de cada faixa, com piso e método provisórios. Este estágio
   também testa descartar a faixa inteira, com uma regra de desempate: **a faixa só fica se
   ganhar do descarte por mais de 0,005 de F1**. Diferença menor que isso é ruído, e cada
   faixa traz o confundidor dela junto.
1. **Quantil de área × método de limiarização.** Mesma margem de 0,005 para empate, e entre
   empatados fica o mais simples.
   **1b.** Parâmetros internos do método vencedor: `blockSize` e `C` se for a adaptativa,
   ou o corte se for o global. Otsu não tem o que ajustar.
2. **Teto de área**, com o resto já fixado. "Sem teto" é um resultado legítimo.

#### O episódio do céu

O teto de área nasceu de uma falha concreta: em imagens de rodovia o pipeline devolvia o
céu como objeto detectado, e o maior deles ocupava 13,6% da cena, passando por todos os
filtros de forma. A causa não era falta de filtro, era a faixa azul, que capturava céu e
quase nenhuma placa. Removida a faixa, o teto voltou a ser o que deveria: filtro de escala
contra fachada e vegetação de perto.

Daí veio a regra de margem do estágio 0. Filtro de forma não corrige faixa de cor sem alvo.


In [ ]:
import itertools

PORTAS_CANDIDATAS = [80, 100, 120, 140, 160, None]   # None = descartar a faixa inteira
PORTA_INICIAL = 110
# Uma faixa de cor so fica no pipeline se render mais que descarta-la por uma margem
# declarada. Sem isso, uma diferenca de milesimo de F1 (ruido da amostra) decide manter
# uma cor, e cada faixa a mais traz o confundidor dela: o azul traz o ceu.
MARGEM_FAIXA = 0.005
QUANTIS_AREA_MIN = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
QUANTIS_AREA_MAX = [0.90, 0.95, 0.99, 1.00]      # 1.00 = sem teto
# A ordem desta lista e o criterio de desempate: do mais simples pro mais complexo.
METODOS_LIMIAR = ["global", "otsu", "otsu_restrito", "adaptativa"]
QUANTIL_PROVISORIO, METODO_PROVISORIO = 0.30, "otsu_restrito"

# --- Estagio 0: porta de saturacao de cada faixa, por descida em coordenadas -
# Uma grade completa custaria len(PORTAS)^len(FAIXAS) avaliacoes. A descida por
# coordenadas percorre uma faixa de cada vez, mantendo as demais fixas, e por isso
# custa apenas len(PORTAS) x len(FAIXAS). O candidato None testa a hipotese
# "esta cor rende menos do que custa em falsos positivos".
linhas = []
area_provisoria = area_minima_para_quantil(ESCALA_AJUSTE, QUANTIL_PROVISORIO)
portas = {f["nome"]: PORTA_INICIAL for f in FAIXAS_MEDIDAS}


def _avaliar_portas(portas_teste: dict) -> dict:
    faixas = faixas_com_portas(FAIXAS_MEDIDAS, portas_teste)
    if not faixas:                       # todas as faixas descartadas: configuracao invalida
        return {"precisao": 0.0, "recall": 0.0, "f1": -1.0, "vp": 0, "fp": 0, "fn": 0,
                "limiar_medio": np.nan, "limiar_desvio": np.nan, "ms_por_imagem": 0.0}
    p = com_parametros(PARAMS, faixas=faixas, area_minima=area_provisoria,
                       area_maxima=0, metodo_limiar=METODO_PROVISORIO)
    return avaliar_configuracao(AMOSTRA_AJUSTE, p)


for nome in [f["nome"] for f in FAIXAS_MEDIDAS]:
    resultados = {}
    for candidato in PORTAS_CANDIDATAS:
        teste = {**portas, nome: candidato}
        metricas = _avaliar_portas(teste)
        resultados[candidato] = metricas["f1"]
        linhas.append({"estagio": 0, "faixa_ajustada": nome,
                       "s_min_testado": -1 if candidato is None else candidato,
                       "portas": ";".join(f"{k}={v}" for k, v in teste.items()),
                       "quantil_area_min": QUANTIL_PROVISORIO, "area_minima": area_provisoria,
                       "quantil_area_max": 1.00, "area_maxima": 0, "metodo": METODO_PROVISORIO,
                       **metricas})
    f1_fora = resultados.get(None, -1.0)
    melhor_porta = max((c for c in resultados if c is not None), key=lambda c: resultados[c])
    ganho = resultados[melhor_porta] - f1_fora
    portas[nome] = melhor_porta if ganho > MARGEM_FAIXA else None
    escolhido = f"s_min = {portas[nome]}" if portas[nome] is not None else "descartada"
    detalhe = "  ".join(f"{'fora' if c is None else c}:{v:.3f}" for c, v in resultados.items())
    print(f"faixa {nome:9s} → {escolhido:14s} | ganho sobre descartar: {ganho:+.3f} "
          f"(margem {MARGEM_FAIXA}) | F1 por candidato: {detalhe}")

GRADE_0 = pd.DataFrame(linhas)
PORTAS_ESCOLHIDAS = dict(portas)
FAIXAS_ESCOLHIDAS = faixas_com_portas(FAIXAS_MEDIDAS, PORTAS_ESCOLHIDAS)
PARAMS = com_parametros(PARAMS, faixas=FAIXAS_ESCOLHIDAS)
F1_ESTAGIO_0 = float(GRADE_0["f1"].max())

print(f"\nESTÁGIO 0 | {len(GRADE_0)} avaliações × {len(AMOSTRA_AJUSTE)} imagens")
print("  faixas mantidas: " + ", ".join(f"{f['nome']} (s_min {f['s_min']})" for f in FAIXAS_ESCOLHIDAS))
descartadas = [n for n, v in PORTAS_ESCOLHIDAS.items() if v is None]
print("  faixas descartadas por medição: " + (", ".join(descartadas) if descartadas else "nenhuma"))

# --- Estagio 1: metodo de limiarizacao x piso de area (ainda sem teto) -------
_portas_txt = ";".join(f"{k}={v}" for k, v in PORTAS_ESCOLHIDAS.items())
linhas = []
for quantil in QUANTIS_AREA_MIN:
    area = area_minima_para_quantil(ESCALA_AJUSTE, quantil)
    for metodo in METODOS_LIMIAR:
        p = com_parametros(PARAMS, area_minima=area, area_maxima=0, metodo_limiar=metodo)
        linhas.append({"estagio": 1, "portas": _portas_txt,
                       "quantil_area_min": quantil, "area_minima": area,
                       "quantil_area_max": 1.00, "area_maxima": 0, "metodo": metodo,
                       **avaliar_configuracao(AMOSTRA_AJUSTE, p)})

GRADE_1 = pd.DataFrame(linhas)

# Metodos dentro da margem do melhor F1 estao empatados: a diferenca entre eles cabe no
# ruido de uma amostra de 250 imagens. Deixar o "vencedor" sair da ordem das linhas seria
# sorte. O desempate e declarado: entre empatados, fica o mais simples, que e o primeiro
# de METODOS_LIMIAR.
_f1_por_metodo = GRADE_1.groupby("metodo")["f1"].max()
EMPATE_METODOS = [m for m in METODOS_LIMIAR
                  if _f1_por_metodo[m] >= _f1_por_metodo.max() - MARGEM_FAIXA]
METODO_ESCOLHIDO = EMPATE_METODOS[0]
_do_metodo = GRADE_1[GRADE_1["metodo"] == METODO_ESCOLHIDO]
melhor_1 = _do_metodo.loc[_do_metodo["f1"].idxmax()]
QUANTIL_ESCOLHIDO = float(melhor_1["quantil_area_min"])
AREA_ESCOLHIDA = int(melhor_1["area_minima"])

print(f"\nESTÁGIO 1 | {len(GRADE_1)} configurações × {len(AMOSTRA_AJUSTE)} imagens")
print(GRADE_1.pivot(index="quantil_area_min", columns="metodo", values="f1").round(3).to_string())
print(f"\n  empate técnico (dentro de {MARGEM_FAIXA} do melhor F1): "
      + ", ".join(f"{m} {_f1_por_metodo[m]:.3f}" for m in EMPATE_METODOS))
print(f"  desempate pelo mais simples → {METODO_ESCOLHIDO}"
      if len(EMPATE_METODOS) > 1 else f"  vencedor isolado: {METODO_ESCOLHIDO}")
print(f"  quantil inferior {QUANTIL_ESCOLHIDO:.2f} → area_minima = {AREA_ESCOLHIDA} px² "
      f"| F1 = {melhor_1['f1']:.3f}")

# --- Estagio 1b: parametros internos do metodo vencedor ---------------------
# O estagio 1 compara os metodos com os parametros internos no valor inicial. Aqui, com o
# metodo e o piso fixados, esses parametros sao ajustados. A janela da adaptativa anda em
# torno do tamanho da placa mediana (~30 px): igual, quase o dobro e mais que o dobro.
PARAMETROS_DO_METODO = {
    "adaptativa": {"adapt_bloco": [31, 51, 71], "adapt_c": [-5, -10, -15]},
    "global": {"limiar_global": [64, 96, 128]},
}
_grade_metodo = PARAMETROS_DO_METODO.get(METODO_ESCOLHIDO, {})
AJUSTE_DO_METODO = {}
linhas = []
for _valores in itertools.product(*_grade_metodo.values()):
    _alteracoes = dict(zip(_grade_metodo, _valores))
    p = com_parametros(PARAMS, metodo_limiar=METODO_ESCOLHIDO, area_minima=AREA_ESCOLHIDA,
                       area_maxima=0, **_alteracoes)
    linhas.append({"estagio": "1b", "portas": _portas_txt,
                   "quantil_area_min": QUANTIL_ESCOLHIDO, "area_minima": AREA_ESCOLHIDA,
                   "quantil_area_max": 1.00, "area_maxima": 0, "metodo": METODO_ESCOLHIDO,
                   **_alteracoes, **avaliar_configuracao(AMOSTRA_AJUSTE, p)})
GRADE_1B = pd.DataFrame(linhas)

if _grade_metodo:
    melhor_1b = GRADE_1B.loc[GRADE_1B["f1"].idxmax()]
    AJUSTE_DO_METODO = {nome: int(melhor_1b[nome]) for nome in _grade_metodo}
    PARAMS = com_parametros(PARAMS, **AJUSTE_DO_METODO)
    print(f"\nESTÁGIO 1b | parâmetros internos de '{METODO_ESCOLHIDO}' ({len(GRADE_1B)} configurações)")
    if len(_grade_metodo) == 2:
        _a, _b = list(_grade_metodo)
        print(GRADE_1B.pivot(index=_a, columns=_b, values="f1").round(3).to_string())
    else:
        print(GRADE_1B[list(_grade_metodo) + ["precisao", "recall", "f1"]].to_string(index=False))
    print(f"  adotado: {AJUSTE_DO_METODO} | F1 = {melhor_1b['f1']:.3f}")
else:
    print(f"\nESTÁGIO 1b | '{METODO_ESCOLHIDO}' não tem parâmetro interno pra ajustar")

# --- Estagio 2: teto de area, com o vencedor do estagio 1 fixado -------------
linhas = []
for quantil_max in QUANTIS_AREA_MAX:
    teto = area_maxima_para_quantil(ESCALA_AJUSTE, quantil_max)
    p = com_parametros(PARAMS, metodo_limiar=METODO_ESCOLHIDO,
                       area_minima=AREA_ESCOLHIDA, area_maxima=teto)
    linhas.append({"estagio": 2, "portas": _portas_txt,
                   "quantil_area_min": QUANTIL_ESCOLHIDO,
                   "area_minima": AREA_ESCOLHIDA, "quantil_area_max": quantil_max,
                   "area_maxima": teto, "metodo": METODO_ESCOLHIDO,
                   **avaliar_configuracao(AMOSTRA_AJUSTE, p)})

GRADE_2 = pd.DataFrame(linhas)
melhor_2 = GRADE_2.loc[GRADE_2["f1"].idxmax()]
QUANTIL_MAX_ESCOLHIDO = float(melhor_2["quantil_area_max"])
AREA_MAXIMA_ESCOLHIDA = int(melhor_2["area_maxima"])
F1_AJUSTE = float(melhor_2["f1"])

print(f"\nESTÁGIO 2 | teto de área (método e piso já fixados)")
print(GRADE_2[["quantil_area_max", "area_maxima", "precisao", "recall", "f1", "vp", "fp", "fn"]]
      .to_string(index=False))

PARAMS = com_parametros(PARAMS, metodo_limiar=METODO_ESCOLHIDO,
                        area_minima=AREA_ESCOLHIDA, area_maxima=AREA_MAXIMA_ESCOLHIDA)

GRADE = pd.concat([GRADE_0, GRADE_1, GRADE_1B, GRADE_2], ignore_index=True)
teto_txt = "sem teto" if AREA_MAXIMA_ESCOLHIDA == 0 else f"{AREA_MAXIMA_ESCOLHIDA} px²"
print(f"\nADOTADO: faixas {_portas_txt} | {METODO_ESCOLHIDO} "
      f"| área ∈ [{AREA_ESCOLHIDA} px², {teto_txt}] | F1 (ajuste) = {F1_AJUSTE:.3f}")

# Melhor configuracao de cada metodo no estagio 1, para a tabela do relatorio.
COMPARACAO_LIMIAR = (GRADE_1.sort_values("f1", ascending=False)
                     .drop_duplicates("metodo").set_index("metodo"))
COMPARACAO_LIMIAR

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(16, 4))

# (a) Estagio 1: trade-off do piso de area, para o metodo vencedor.
curva = GRADE_1[GRADE_1["metodo"] == METODO_ESCOLHIDO].sort_values("area_minima")
eixos[0].plot(curva["area_minima"], curva["precisao"], "o-", color="#2d6a9f", label="precisão")
eixos[0].plot(curva["area_minima"], curva["recall"], "s-", color="#e08214", label="recall")
eixos[0].plot(curva["area_minima"], curva["f1"], "^-", color="#c0392b", lw=2.2, label="F1")
eixos[0].axvline(AREA_ESCOLHIDA, color="#4d9078", ls="--", lw=2,
                 label=f"adotado: {AREA_ESCOLHIDA} px²")
eixos[0].set_xscale("log")
eixos[0].set_xlabel("piso de área do contorno (px², escala log)")
eixos[0].set_title(f"(a) Estágio 1: piso de área · {METODO_ESCOLHIDO}")
eixos[0].legend(fontsize=8); eixos[0].grid(alpha=0.3)

# (b) Estagio 2: efeito do teto de area. E aqui que o ceu sai da contagem.
rotulos = [("sem teto" if q >= 1.0 else f"q={q:.2f}") for q in GRADE_2["quantil_area_max"]]
x = np.arange(len(rotulos))
eixos[1].plot(x, GRADE_2["precisao"], "o-", color="#2d6a9f", label="precisão")
eixos[1].plot(x, GRADE_2["recall"], "s-", color="#e08214", label="recall")
eixos[1].plot(x, GRADE_2["f1"], "^-", color="#c0392b", lw=2.2, label="F1")
eixos[1].set_xticks(x); eixos[1].set_xticklabels(rotulos, fontsize=8)
eixos[1].set_xlabel("teto de área (quantil superior das placas anotadas)")
eixos[1].set_title("(b) Estágio 2: teto de área (fachadas, vegetação)")
eixos[1].legend(fontsize=8); eixos[1].grid(alpha=0.3)

# (c) Comparacao entre metodos de limiarizacao.
COMPARACAO_LIMIAR[["precisao", "recall", "f1"]].plot(
    kind="bar", ax=eixos[2], color=["#2d6a9f", "#e08214", "#4d9078"], rot=15, width=0.78)
eixos[2].set_title(f"(c) Melhor configuração de cada método (adotado: {METODO_ESCOLHIDO})")
eixos[2].set_ylabel("métrica"); eixos[2].legend(fontsize=8); eixos[2].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(DIR_FIGURAS / "01_escolha_de_parametros.png", bbox_inches="tight")
plt.show()

---
## 6. Ablação do pré-processamento

Com os parâmetros fixados, mede-se o efeito de desligar o CLAHE e a suavização. Responde
com número ao primeiro erro apontado na orientação da AED, que é "segmentar sem suavizar e
concluir que Otsu não funciona". Roda na amostra de ajuste, porque é diagnóstico.

**Como ler:** compare os pares de mesma suavização para avaliar o CLAHE, e as três
suavizações com CLAHE ligado para avaliar o filtro. Como o kernel já é o menor que a escala
permite, a diferença tende a ser pequena.

A célula imprime as conclusões calculadas da tabela, para o relatório não afirmar mais do
que a medição sustenta.


In [ ]:
linhas = []
for usar_clahe in (True, False):
    for suavizacao in ("gaussiano", "mediana", "nenhuma"):
        p = com_parametros(PARAMS, usar_clahe=usar_clahe, suavizacao=suavizacao)
        linhas.append({"clahe": usar_clahe, "suavizacao": suavizacao,
                       **avaliar_configuracao(AMOSTRA_AJUSTE, p)})

ABLACAO = pd.DataFrame(linhas).sort_values("f1", ascending=False).reset_index(drop=True)
print(f"ABLAÇÃO | método '{METODO_ESCOLHIDO}', faixas {_portas_txt}, "
      f"área mínima {PARAMS.area_minima} px², suavização {PARAMS.suavizacao_k}×{PARAMS.suavizacao_k}\n")
print(ABLACAO.to_string(index=False))

# As conclusoes abaixo sao calculadas da tabela, nao escritas na mao.
_f1 = ABLACAO.set_index(["clahe", "suavizacao"])["f1"]
ganhos_clahe = {s: round(float(_f1[(True, s)] - _f1[(False, s)]), 3) for s in ("gaussiano", "mediana", "nenhuma")}
_com_clahe = ABLACAO[ABLACAO["clahe"]].sort_values("f1", ascending=False)
melhor_suavizacao = str(_com_clahe.iloc[0]["suavizacao"])
f1_gauss = float(_f1[(True, "gaussiano")])
CONCLUSOES_ABLACAO = {
    "ganho_f1_clahe_por_suavizacao": ganhos_clahe,
    "clahe_ajuda_em_todos_os_pares": bool(all(g > 0 for g in ganhos_clahe.values())),
    "melhor_suavizacao_com_clahe": melhor_suavizacao,
    "f1_gaussiano": round(f1_gauss, 3),
    "diferenca_melhor_menos_gaussiano": round(float(_com_clahe.iloc[0]["f1"]) - f1_gauss, 3),
}
print("\nCONCLUSÕES CALCULADAS")
print(f"  CLAHE: ganho de F1 por suavização = {ganhos_clahe} → "
      f"{'ajuda em todos os pares' if CONCLUSOES_ABLACAO['clahe_ajuda_em_todos_os_pares'] else 'NÃO ajuda em todos os pares'}")
print(f"  Suavização (com CLAHE): melhor = '{melhor_suavizacao}'; "
      f"diferença para o gaussiano adotado = {CONCLUSOES_ABLACAO['diferenca_melhor_menos_gaussiano']:+.3f}")

fig, eixo = plt.subplots(figsize=(7.5, 3.6))
rotulos = [f"CLAHE={'sim' if r.clahe else 'não'}\n{r.suavizacao}" for r in ABLACAO.itertuples()]
cores = ["#4d9078" if (r.clahe == PARAMS.usar_clahe and r.suavizacao == PARAMS.suavizacao)
         else "#2d6a9f" for r in ABLACAO.itertuples()]
eixo.bar(rotulos, ABLACAO["f1"], color=cores)
eixo.set_title("Ablação do pré-processamento (F1). Verde: configuração adotada")
eixo.set_ylabel("F1"); eixo.tick_params(axis="x", labelsize=7.5); eixo.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(DIR_FIGURAS / "02_ablacao_preprocessamento.png", bbox_inches="tight")
plt.show()

### 6.1. O histograma que justifica a limiarização

Para uma imagem representativa, o histograma do mapa de evidência com o corte de cada
método sobreposto. Deixa visível a patologia da Seção 3.6: o pico em zero, que é o fundo,
puxa o Otsu clássico para baixo, enquanto o Otsu restrito corta onde placa e fundo de fato
se separam.


In [ ]:
def diagramar_histograma(caminho: Path, params: Parametros, salvar: Path | None = None):
    """Histograma do mapa de evidencia com os limiares de cada metodo sobrepostos."""
    resultado = executar_pipeline(caminho, params)
    mapa = resultado["etapas"]["4_evidencia"]
    limiares = {m: limiarizar(mapa, m, params.limiar_global)[1]
                for m in ("global", "otsu", "otsu_restrito")}

    fig, eixos = plt.subplots(1, 3, figsize=(14, 3.6))
    eixos[0].imshow(resultado["etapas"]["1_original"]); eixos[0].axis("off")
    eixos[0].set_title("Imagem de referência")
    eixos[1].imshow(mapa, cmap="magma", vmin=0, vmax=255); eixos[1].axis("off")
    eixos[1].set_title("Mapa de evidência cromática")

    eixos[2].hist(mapa.ravel(), bins=64, range=(0, 255), color="#7f8c8d", log=True)
    cores = {"global": "#2d6a9f", "otsu": "#e08214", "otsu_restrito": "#c0392b"}
    for nome, valor in limiares.items():
        if not np.isnan(valor):
            eixos[2].axvline(valor, color=cores[nome], lw=2, ls="--",
                             label=f"{nome} = {valor:.0f}")
    eixos[2].set_title("Histograma do mapa (escala log)")
    eixos[2].set_xlabel("evidência cromática"); eixos[2].set_ylabel("pixels")
    eixos[2].legend(fontsize=8)

    fig.tight_layout()
    if salvar:
        fig.savefig(salvar, bbox_inches="tight")
    return fig


diagramar_histograma(AMOSTRA_AJUSTE[0], PARAMS, DIR_FIGURAS / "03_histograma_limiares.png")
plt.show()

---
## 7. Evidências visuais, antes e depois de cada etapa

As imagens são sorteadas com semente fixa entre as do **lado de validação**, as mesmas que
geram o número da Seção 8. Sem escolher na mão os casos favoráveis: nesta execução, duas
das seis não produzem nenhuma detecção.

Cada painel percorre as seis etapas, mostra o Canny (que não alimenta a contagem) e termina
com as detecções sobre a imagem de entrada. Depois vem a tabela de T por imagem, no formato
da receita da Apostila 02, salva em `outputs/limiares_por_imagem.csv`.


In [ ]:
# Sorteadas do lado de VALIDACAO, e nao da base inteira: assim o painel visual e o numero
# reportado na Secao 8 falam do mesmo conjunto, que nao participou de nenhuma decisao.
AMOSTRA_EVIDENCIAS = [Path(p) for p in POOL_VALIDACAO.sample(
    min(N_EVIDENCIAS_VISUAIS, len(POOL_VALIDACAO)), random_state=SEMENTE + 1)["caminho"]]

RESULTADOS_EVIDENCIA = []
for i, caminho in enumerate(AMOSTRA_EVIDENCIAS, 1):
    resultado = executar_pipeline(caminho, PARAMS)
    RESULTADOS_EVIDENCIA.append((caminho, resultado))
    painel_pipeline(resultado, titulo=f"[{i}/{len(AMOSTRA_EVIDENCIAS)}] {caminho.name}",
                    salvar=DIR_FIGURAS / f"04_pipeline_{i:02d}_{caminho.stem[:40]}.jpg")
    plt.show()

In [ ]:
# Formato da receita da Apostila 02: T e pixels de objeto de cada imagem. A segmentacao
# usa a limiarizacao adaptativa, que nao tem um T unico, entao o T abaixo e o do Otsu
# restrito calculado sobre o mesmo mapa de evidencia. Ele nao muda a mascara, serve como
# diagnostico: a variacao dele entre imagens mostra a instabilidade de iluminacao.
def limiar_diagnostico(resultado: dict) -> float:
    return limiarizar(resultado["etapas"]["4_evidencia"], "otsu_restrito")[1]


linhas = []
for caminho, resultado in RESULTADOS_EVIDENCIA:
    T = limiar_diagnostico(resultado)
    pixels = int(resultado["etapas"]["6_morfologia"].sum() / 255)
    print(f"{caminho.name[:44]:<44} Limiar de Otsu: {T:5.1f} | pixels de objeto: {pixels}")
    linhas.append({"imagem": caminho.name, "limiar_T": T, "pixels_objeto": pixels,
                   "objetos": len(resultado["objetos"])})

LIMIARES_EVIDENCIA = pd.DataFrame(linhas)
LIMIARES_EVIDENCIA.to_csv(DIR_SAIDA / "limiares_por_imagem.csv", index=False, encoding="utf-8")
_T = LIMIARES_EVIDENCIA.loc[LIMIARES_EVIDENCIA["limiar_T"] < 255, "limiar_T"]   # 255 = sem evidencia
print(f"\nT medio {_T.mean():.1f} | desvio {_T.std():.1f} | de {_T.min():.1f} a {_T.max():.1f}")


In [ ]:
def mosaico_deteccoes(resultados: list, salvar: Path | None = None):
    """Grade com o resultado final de todas as imagens de evidencia lado a lado."""
    n = len(resultados)
    colunas = min(3, n)
    linhas = math.ceil(n / colunas)
    fig, eixos = plt.subplots(linhas, colunas, figsize=(5.2 * colunas, 3.9 * linhas))
    eixos = np.atleast_1d(eixos).ravel()
    for eixo, (caminho, resultado) in zip(eixos, resultados):
        eixo.imshow(desenhar_deteccoes(resultado))
        eixo.set_title(f"{caminho.name[:34]}\ndetectados: {len(resultado['objetos'])}  ·  "
                       f"anotados: {len(ler_rotulos(caminho))}", fontsize=8)
        eixo.axis("off")
    for eixo in eixos[n:]:
        eixo.axis("off")
    fig.suptitle("Saída do pipeline: contagem e classe geométrica por objeto", y=1.0)
    fig.tight_layout()
    if salvar:
        salvar_figura(fig, salvar)
    return fig


mosaico_deteccoes(RESULTADOS_EVIDENCIA, DIR_FIGURAS / "05_mosaico_deteccoes.jpg")
plt.show()

In [ ]:
# Tabela de descritores, que e a saida numerica que responde ao problema.
linhas = []
for caminho, resultado in RESULTADOS_EVIDENCIA:
    for i, obj in enumerate(resultado["objetos"], 1):
        linhas.append({"imagem": caminho.name, "objeto": i, "forma": obj["forma"],
                       "ambigua": obj["forma_ambigua"], "vertices": obj["vertices"],
                       "area_px2": round(obj["area_px"]),
                       "perimetro_px": round(obj["perimetro_px"], 1),
                       "centroide_x": round(obj["cx"], 1), "centroide_y": round(obj["cy"], 1),
                       "largura": obj["w"], "altura": obj["h"],
                       "razao_aspecto": round(obj["razao_aspecto"], 3),
                       "circularidade": round(obj["circularidade"], 3),
                       "solidez": round(obj["solidez"], 3),
                       "extensao": round(obj["extensao"], 3),
                       **{f"hu{k}": round(obj[f"hu{k}"], 3) for k in range(1, 8)}})

DESCRITORES = pd.DataFrame(linhas)
DESCRITORES.to_csv(DIR_SAIDA / "descritores_objetos.csv", index=False, encoding="utf-8")
print(f"{len(DESCRITORES)} objetos descritos em {len(RESULTADOS_EVIDENCIA)} imagens\n")
DESCRITORES

In [ ]:
# Sao estes recortes que vao alimentar a CNN da 2a Etapa.
recortes = [r for _, resultado in RESULTADOS_EVIDENCIA for r in recortar_regioes(resultado)][:12]
if recortes:
    colunas = min(6, len(recortes))
    linhas_grade = math.ceil(len(recortes) / colunas)
    fig, eixos = plt.subplots(linhas_grade, colunas, figsize=(1.9 * colunas, 2.1 * linhas_grade))
    eixos = np.atleast_1d(eixos).ravel()
    for eixo, recorte in zip(eixos, recortes):
        eixo.imshow(cv2.resize(recorte, (96, 96), interpolation=cv2.INTER_CUBIC))
        eixo.axis("off")
    for eixo in eixos[len(recortes):]:
        eixo.axis("off")
    fig.suptitle("ROIs extraídas e normalizadas (entrada da etapa de IA na N2)", y=1.01)
    fig.tight_layout()
    salvar_figura(fig, DIR_FIGURAS / "06_rois_normalizadas.jpg")
    plt.show()
else:
    print("Nenhuma ROI extraida nesta amostra.")

---
## 8. Avaliação quantitativa na amostra de validação retida

Nenhuma imagem daqui participou da escolha de parâmetro, então o número é estimativa
honesta. Duas leituras:

- **Detecção:** precisão, recall e F1 por casamento `IoU ≥ 0,30`. Mede se as placas foram
  localizadas.
- **Contagem:** erro absoluto médio entre detectados e anotados. É a métrica do caso de uso
  de inventário, mas aqui ela precisa de referência: quase toda imagem avaliada tem
  exatamente uma placa anotada, então chutar "1" já acerta muito. A célula faz essa
  comparação antes de qualquer conclusão.

O resultado não é alto, e isso é informação. Um detector puramente cromático é a linha de
base contra a qual o modelo treinado da 2ª Etapa será comparado.


In [ ]:
def avaliar_em_detalhe(caminhos: list[Path], params: Parametros,
                       limiar_iou: float = 0.30) -> pd.DataFrame:
    """Avaliacao imagem a imagem: detectados, anotados, VP/FP/FN e erro de contagem."""
    linhas = []
    for caminho in caminhos:
        resultado = executar_pipeline(caminho, params)
        altura, largura = resultado["dimensoes_trabalho"]
        detectadas = [(o["x"], o["y"], o["w"], o["h"]) for o in resultado["objetos"]]
        anotadas_img = caixas_anotadas(caminho, largura, altura)
        vp, fp, fn = casar_deteccoes(detectadas, anotadas_img, limiar_iou)
        linhas.append({"imagem": caminho.name, "limiar_T": limiar_diagnostico(resultado),
                       "detectados": len(detectadas), "anotados": len(anotadas_img),
                       "vp": vp, "fp": fp, "fn": fn,
                       "erro_contagem": len(detectadas) - len(anotadas_img)})
    return pd.DataFrame(linhas)


AVALIACAO = avaliar_em_detalhe(AMOSTRA_VALIDACAO, PARAMS)

# T = 255 e o sentinela de "imagem sem pixel com evidencia de cor" (mascara vazia).
# Nao e um limiar de verdade, entao fica fora da media e do desvio e e contado a parte.
_T_validos = AVALIACAO.loc[AVALIACAO["limiar_T"] < 255, "limiar_T"]

vp, fp, fn = int(AVALIACAO["vp"].sum()), int(AVALIACAO["fp"].sum()), int(AVALIACAO["fn"].sum())
precisao = vp / (vp + fp) if vp + fp else 0.0
recall = vp / (vp + fn) if vp + fn else 0.0
f1 = 2 * precisao * recall / (precisao + recall) if precisao + recall else 0.0

RESUMO_AVALIACAO = {
    "amostra": "validacao retida (disjunta da amostra de ajuste)",
    "imagens_avaliadas": len(AVALIACAO),
    "objetos_anotados": int(AVALIACAO["anotados"].sum()),
    "objetos_detectados": int(AVALIACAO["detectados"].sum()),
    "verdadeiros_positivos": vp, "falsos_positivos": fp, "falsos_negativos": fn,
    "precisao": round(precisao, 3), "recall": round(recall, 3), "f1": round(f1, 3),
    "limiar_iou": 0.30,
    "erro_absoluto_medio_contagem": round(float(AVALIACAO["erro_contagem"].abs().mean()), 2),
    "imagens_com_contagem_exata": int((AVALIACAO["erro_contagem"] == 0).sum()),
    "limiar_T_medio": round(float(_T_validos.mean()), 1),
    "limiar_T_desvio": round(float(_T_validos.std()), 1),
    "limiar_T_minimo": round(float(_T_validos.min()), 1),
    "limiar_T_maximo": round(float(_T_validos.max()), 1),
    "imagens_sem_evidencia_cromatica": int((AVALIACAO["limiar_T"] >= 255).sum()),
    "f1_na_amostra_de_ajuste": round(F1_AJUSTE, 3),
    "diferenca_ajuste_validacao": round(F1_AJUSTE - f1, 3),
}

# Referencia pra contagem: chutar sempre o numero de placas mais comum no lado de AJUSTE.
# Se o pipeline nao bate esse chute, o erro de contagem nao mostra merito nenhum.
CHUTE_CONSTANTE = int(POOL_AJUSTE["n_objetos"].mode().iloc[0])
_erro_chute = (AVALIACAO["anotados"] - CHUTE_CONSTANTE).abs()
RESUMO_AVALIACAO["chute_constante_de_contagem"] = CHUTE_CONSTANTE
RESUMO_AVALIACAO["erro_absoluto_medio_chute_constante"] = round(float(_erro_chute.mean()), 2)
RESUMO_AVALIACAO["contagem_exata_chute_constante"] = int((_erro_chute == 0).sum())

print("DESEMPENHO DA LINHA DE BASE CLASSICA (PDI, sem aprendizado)")
print("=" * 62)
for chave, valor in RESUMO_AVALIACAO.items():
    print(f"{chave:<34} {valor}")
print("=" * 62)
print(f"\nF1 na amostra de ajuste     : {F1_AJUSTE:.3f}  (onde os parametros foram escolhidos)")
print(f"F1 na amostra de validacao  : {f1:.3f}  (retida, estimativa honesta)")
if F1_AJUSTE - f1 > 0.05:
    print("\nA diferenca entre as duas mede o otimismo da selecao de parametros: escolher\n"
          "entre varias configuracoes na mesma amostra em que se mede inflaria o resultado.\n"
          "E exatamente esse vies que a amostra retida evita.")
print(f"\nContagem: erro medio {RESUMO_AVALIACAO['erro_absoluto_medio_contagem']} do pipeline contra "
      f"{RESUMO_AVALIACAO['erro_absoluto_medio_chute_constante']} de quem chuta sempre "
      f"{CHUTE_CONSTANTE} placa(s).")
if RESUMO_AVALIACAO["erro_absoluto_medio_contagem"] >= RESUMO_AVALIACAO["erro_absoluto_medio_chute_constante"]:
    print("O pipeline conta pior que o chute constante. Como quase toda imagem avaliada tem\n"
          "exatamente uma placa anotada, o erro de contagem nao serve pra avaliar o pipeline\n"
          "nesta base. A leitura que vale e a de deteccao (precisao, recall e F1).")
AVALIACAO.head(15)

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12.5, 3.6))

eixos[0].bar(["precisão", "recall", "F1"], [precisao, recall, f1],
             color=["#2d6a9f", "#e08214", "#4d9078"])
eixos[0].set_ylim(0, 1); eixos[0].grid(axis="y", alpha=0.3)
eixos[0].set_title(f"Detecção · IoU ≥ 0,30 · {len(AVALIACAO)} imagens retidas")
for i, valor in enumerate([precisao, recall, f1]):
    eixos[0].text(i, valor + 0.02, f"{valor:.2f}", ha="center", fontsize=9)

limite = max(3, int(AVALIACAO["erro_contagem"].abs().max()))
eixos[1].hist(AVALIACAO["erro_contagem"], bins=np.arange(-limite - 0.5, limite + 1.5),
              color="#2d6a9f", edgecolor="white")
eixos[1].axvline(0, color="#c1440e", lw=2, label="contagem exata")
eixos[1].set_title("Erro de contagem (detectados − anotados)")
eixos[1].set_xlabel("erro"); eixos[1].legend(fontsize=8); eixos[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(DIR_FIGURAS / "07_avaliacao_quantitativa.png", bbox_inches="tight")
plt.show()

In [ ]:
# Distribuicao das classes geometricas atribuidas em toda a amostra de validacao.
formas = []
for caminho in AMOSTRA_VALIDACAO:
    for obj in executar_pipeline(caminho, PARAMS)["objetos"]:
        formas.append({"forma": obj["forma"], "ambigua": obj["forma_ambigua"]})

if formas:
    DISTRIBUICAO_FORMAS = (pd.DataFrame(formas)
                           .groupby("forma")
                           .agg(objetos=("forma", "size"), ambiguos=("ambigua", "sum"))
                           .sort_values("objetos", ascending=False))
    DISTRIBUICAO_FORMAS["% do total"] = (DISTRIBUICAO_FORMAS["objetos"] /
                                         DISTRIBUICAO_FORMAS["objetos"].sum() * 100).round(1)
    print("CLASSIFICACAO GEOMETRICA NA AMOSTRA DE VALIDACAO\n")
    print(DISTRIBUICAO_FORMAS.to_string())
else:
    DISTRIBUICAO_FORMAS = pd.DataFrame()
    print("Nenhum objeto detectado na amostra.")

---
## 9. Diagrama da arquitetura da solução

Da imagem de entrada até a saída pretendida, com o ponto em que o modelo de IA entra nas
próximas etapas.

A fronteira entre as duas etapas é a ROI normalizada: o Checkpoint 1 entrega o recorte com
os descritores, e é esse recorte que a 2ª Etapa consome. A correção de iluminação e o filtro
por área seguem como pré e pós-processamento do detector treinado.


In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


def _caixa(eixo, x, y, largura, altura, texto, cor_fundo, cor_borda,
           tamanho=8.0, estilo="solid", peso="normal"):
    eixo.add_patch(FancyBboxPatch((x, y), largura, altura,
                                  boxstyle="round,pad=0.02,rounding_size=0.12",
                                  linewidth=1.3, edgecolor=cor_borda, facecolor=cor_fundo,
                                  linestyle=estilo, zorder=2))
    eixo.text(x + largura / 2, y + altura / 2, texto, ha="center", va="center",
              fontsize=tamanho, color="#1b262c", zorder=3, fontweight=peso, linespacing=1.35)


def _seta(eixo, origem, destino, estilo="solid", cor="#5a6b7b"):
    eixo.add_patch(FancyArrowPatch(origem, destino, arrowstyle="-|>", mutation_scale=13,
                                   linewidth=1.3, color=cor, linestyle=estilo, zorder=1))


def desenhar_diagrama_arquitetura(params: Parametros, salvar: Path | None = None):
    """Diagrama de fluxo do sistema, da imagem de entrada a saida da 2a Etapa."""
    faixas = [
        ("1 · AQUISIÇÃO", "#eef4f9", "#2d6a9f", [
            f"Dataset Roboflow\n{PROJETO} v{VERSAO}",
            f"Inventário automático\n{len(INVENTARIO)} imagens · {len(CLASSES)} classes",
            "Anotações YOLO\n(escala real do objeto)"]),
        ("2 · PRÉ-PROCESSAMENTO", "#fdf3e7", "#e08214", [
            f"Redimensionar\nlargura = {params.largura_trabalho} px",
            f"CLAHE no canal L* (LAB)\nclip = {params.clahe_clip} · grade {params.clahe_grade}×{params.clahe_grade}",
            f"Suavização {params.suavizacao}\nkernel {params.suavizacao_k}×{params.suavizacao_k}"]),
        ("3 · SEGMENTAÇÃO", "#eef7f2", "#4d9078", [
            "Mapa de evidência\ncromática (HSV, CONTRAN)",
            f"Limiarização\n{params.metodo_limiar}",
            f"Abertura {params.k_abertura}×{params.k_abertura} → fechamento "
            f"{params.k_fechamento}×{params.k_fechamento}\n→ preenchimento"]),
        ("4 · EXTRAÇÃO", "#f4eef7", "#8e5aa8", [
            "findContours\nRETR_EXTERNAL",
            (f"Filtros · área ∈ [{params.area_minima}, "
             f"{params.area_maxima if params.area_maxima else '∞'}] px²\naspecto · extensão · solidez"),
            "Descritores geométricos\ne classificação de forma"]),
        ("5 · SAÍDA DO CHECKPOINT 1", "#fcecec", "#c0392b", [
            "Contagem · área\ncentroide · caixa",
            "Classe geométrica\n(circular, losango, ...)",
            "ROI normalizada\n+ CSV de descritores"]),
    ]

    fig, eixo = plt.subplots(figsize=(13.5, 10.6))
    eixo.set_xlim(0, 13.5); eixo.set_ylim(-0.45, 10.25); eixo.axis("off")

    x0, largura_caixa, vao = 2.35, 3.25, 0.35
    altura_caixa, altura_faixa = 0.95, 1.55
    topo = 9.55

    centros_faixa = []
    for i, (titulo, fundo, borda, caixas) in enumerate(faixas):
        y = topo - (i + 1) * altura_faixa
        eixo.add_patch(FancyBboxPatch((0.25, y - 0.12), 13.0, altura_faixa - 0.18,
                                      boxstyle="round,pad=0.02,rounding_size=0.1",
                                      linewidth=0, facecolor=fundo, zorder=0))
        eixo.text(0.45, y + altura_caixa / 2 - 0.12, titulo.replace(" · ", "\n"),
                  fontsize=8.5, fontweight="bold", color=borda, va="center", linespacing=1.5)
        for j, texto in enumerate(caixas):
            x = x0 + j * (largura_caixa + vao)
            _caixa(eixo, x, y, largura_caixa, altura_caixa, texto, "white", borda)
            if j:
                _seta(eixo, (x - vao, y + altura_caixa / 2), (x - 0.02, y + altura_caixa / 2))
        centros_faixa.append((y, y + altura_caixa))
        if i:
            y_ant = centros_faixa[i - 1][0]
            _seta(eixo, (x0 + largura_caixa / 2, y_ant - 0.02),
                  (x0 + largura_caixa / 2, y + altura_caixa + 0.02))

    # Faixa da 2a Etapa: tracejada, indicando o que ainda nao foi implementado.
    y = topo - 6 * altura_faixa - 0.25
    eixo.add_patch(FancyBboxPatch((0.25, y - 0.12), 13.0, altura_faixa - 0.18,
                                  boxstyle="round,pad=0.02,rounding_size=0.1",
                                  linewidth=1.4, linestyle="--", edgecolor="#5a6b7b",
                                  facecolor="#f4f6f7", zorder=0))
    eixo.text(0.45, y + altura_caixa / 2 - 0.12, "6 · 2ª ETAPA\n(N2, com IA)",
              fontsize=8.5, fontweight="bold", color="#34495e", va="center", linespacing=1.5)
    for j, texto in enumerate(["Detector YOLO\ntreinado em cena completa",
                               "CNN classificadora\n(GTSRB · 43 classes)",
                               "Classe da placa + laudo\naval. por mAP, IoU e acurácia"]):
        x = x0 + j * (largura_caixa + vao)
        _caixa(eixo, x, y, largura_caixa, altura_caixa, texto, "white", "#5a6b7b", estilo="--")
        if j:
            _seta(eixo, (x - vao, y + altura_caixa / 2), (x - 0.02, y + altura_caixa / 2), "--")

    x_roi = x0 + 2 * (largura_caixa + vao) + largura_caixa / 2
    _seta(eixo, (x_roi, centros_faixa[-1][0] - 0.02), (x_roi, y + altura_caixa + 0.02), "--", "#c0392b")
    eixo.text(x_roi - 0.18, y + altura_caixa + 0.16,
              "a ROI normalizada é a fronteira entre as duas etapas",
              fontsize=7.5, style="italic", color="#c0392b", va="bottom", ha="right")

    eixo.text(6.85, 9.95, "Sistema Inteligente de Detecção e Classificação de Placas de Trânsito",
              ha="center", fontsize=12, fontweight="bold", color="#1b262c")
    eixo.text(6.85, 9.66, "Arquitetura da solução: pipeline de PDI clássico (N1) e "
                          "extensão por aprendizado profundo (N2)",
              ha="center", fontsize=8.5, color="#5a6b7b")

    fig.tight_layout()
    if salvar:
        fig.savefig(salvar, bbox_inches="tight", facecolor="white")
    return fig


desenhar_diagrama_arquitetura(PARAMS, DIR_DOCS / "arquitetura_pipeline.png")
plt.show()

---
## 10. Registro dos parâmetros e exportação dos artefatos

Gera o registro dos parâmetros adotados (que o enunciado pede no README), as tabelas em
CSV, as figuras e um `.zip` com o conjunto.


In [ ]:
REGISTRO = {
    "projeto": "Sistema Inteligente de Deteccao e Classificacao de Placas de Transito",
    "etapa": "AED 2a Etapa - Checkpoint 1 (N1)",
    "disciplina": "CDI1021 - Visao Computacional (2026/2) - PUC Goias",
    "gerado_em": time.strftime("%Y-%m-%d %H:%M:%S"),
    "ambiente": {"python": sys.version.split()[0], "opencv": cv2.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__, "semente": SEMENTE},
    "dataset": {
        "fonte": f"Roboflow Universe / {WORKSPACE}/{PROJETO} versao {VERSAO}",
        "url": f"https://universe.roboflow.com/{WORKSPACE}/{PROJETO}/dataset/{VERSAO}",
        "local": str(DIR_DATASET), "imagens": int(len(INVENTARIO)),
        "objetos_anotados": int(INVENTARIO["n_objetos"].sum()),
        "classes": int(len(CLASSES)),
        "dimensao_mediana": (f"{int(dim['largura'].median())}x{int(dim['altura'].median())}"
                             if len(dim) else "n/d"),
    },
    "parametros_adotados": PARAMS.para_dict(),
    "memoria_de_calculo": MEMORIA_CALIBRACAO,
    "protocolo": {
        "amostra_ajuste": len(AMOSTRA_AJUSTE),
        "amostra_validacao": len(AMOSTRA_VALIDACAO),
        "disjuntas": True,
        "divisao": DIVISAO,
        "observacao": ("parametros escolhidos na amostra de ajuste; metricas reportadas "
                       "na amostra de validacao, que nao participou de nenhuma decisao"),
    },
    "calibracao_cromatica": {
        "amostra": "ajuste",
        "perfil_por_classe": json.loads(PERFIL_CLASSES.to_json(orient="records")),
        "formacao_das_faixas": json.loads(RESUMO_FAIXAS.to_json(orient="records")),
        "faixas_medidas": FAIXAS_MEDIDAS,
        "sobrevivencia_por_porta": json.loads(TABELA_PORTAS.reset_index().to_json(orient="records")),
    },
    "escolha_de_parametros": {
        "portas_de_saturacao": {k: v for k, v in PORTAS_ESCOLHIDAS.items()},
        "margem_para_manter_faixa": MARGEM_FAIXA,
        "faixas_adotadas": PARAMS.faixas,
        "metodo_escolhido": METODO_ESCOLHIDO,
        "empate_tecnico_entre_metodos": EMPATE_METODOS,
        "criterio_de_desempate": ("metodos dentro de {:.3f} de F1 do melhor sao considerados "
                                  "empatados; entre eles fica o mais simples, na ordem "
                                  "declarada {}").format(MARGEM_FAIXA, METODOS_LIMIAR),
        "quantil_area_min": QUANTIL_ESCOLHIDO,
        "area_minima_px2": AREA_ESCOLHIDA,
        "quantil_area_max": QUANTIL_MAX_ESCOLHIDO,
        "area_maxima_px2": AREA_MAXIMA_ESCOLHIDA,
        "f1_na_amostra_de_ajuste": round(F1_AJUSTE, 3),
        "criterio": (f"busca por coordenadas em 4 estagios (0, 1, 1b e 2), {len(GRADE)} configuracoes sobre "
                     f"{len(AMOSTRA_AJUSTE)} imagens de ajuste, maior F1 com IoU >= 0.30"),
        "limiar_T_diagnostico": ("T do Otsu restrito calculado sobre o mapa de evidencia de "
                                 "cada imagem; nao entra na segmentacao, que usa o metodo "
                                 "escolhido acima"),
        "parametros_do_metodo_calibrados": AJUSTE_DO_METODO,
        "valores_fixos_nao_calibrados": {
            nome: getattr(PARAMS, nome) for nome in ("limiar_global", "adapt_bloco", "adapt_c")
            if nome not in AJUSTE_DO_METODO
        },
        "limiares_por_imagem_evidencia": json.loads(LIMIARES_EVIDENCIA.to_json(orient="records")),
        "grade": json.loads(GRADE.to_json(orient="records")),
        "melhor_por_metodo": json.loads(COMPARACAO_LIMIAR.reset_index().to_json(orient="records")),
    },
    "ablacao_preprocessamento": json.loads(ABLACAO.to_json(orient="records")),
    "conclusoes_ablacao": CONCLUSOES_ABLACAO,
    "desempenho": RESUMO_AVALIACAO,
}

arquivo_json = DIR_SAIDA / "parametros_adotados.json"
arquivo_json.write_text(json.dumps(REGISTRO, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Registro salvo em {arquivo_json}")
print(json.dumps(REGISTRO["parametros_adotados"], indent=2, ensure_ascii=False))

In [ ]:
def tabela_markdown_parametros(registro: dict) -> str:
    """Gera a secao de parametros pronta para colar no README.md do repositorio."""
    p = registro["parametros_adotados"]
    memoria = registro["memoria_de_calculo"]
    escolha = registro["escolha_de_parametros"]
    protocolo = registro["protocolo"]
    d = registro["dataset"]
    desempenho = registro["desempenho"]
    faixas_txt = ", ".join("{} (H {:.0f} +/- {:.0f})".format(f["nome"], f["h_centro"], f["h_sigma"])
                           for f in p["faixas"])
    portas_txt = ", ".join("{} {}".format(f["nome"], f["s_min"]) for f in p["faixas"])
    fora = [n for n, v in escolha["portas_de_saturacao"].items() if v is None]
    calibrados = escolha.get("parametros_do_metodo_calibrados", {})
    texto_c = f"O pixel precisa ficar {-p['adapt_c']} níveis acima da média local"

    def origem_interna(nome: str, uso: str) -> str:
        if nome in calibrados:
            return f"Estágio 1b da busca em grade. {uso}"
        if nome.startswith("adapt") == (escolha["metodo_escolhido"] == "adaptativa"):
            return f"Valor fixo, não passou pela busca em grade. {uso}"
        return "Só entra na comparação de métodos. Valor fixo, não calibrado"

    linhas = [
        "## Parâmetros adotados",
        "",
        f"Gerado automaticamente pelo notebook em {registro['gerado_em']} "
        f"(semente {registro['ambiente']['semente']}, OpenCV {registro['ambiente']['opencv']}, "
        f"Python {registro['ambiente']['python']}).",
        "",
        "| Parâmetro | Valor | Origem |",
        "|---|---|---|",
        f"| Largura de trabalho | {p['largura_trabalho']} px | Deixa os parâmetros em pixel comparáveis entre imagens |",
        f"| CLAHE (clip / grade) | {p['clahe_clip']} / {p['clahe_grade']}×{p['clahe_grade']} | Canal L* do LAB, preserva a matiz |",
        f"| Suavização | {p['suavizacao']} {p['suavizacao_k']}×{p['suavizacao_k']} | `{memoria.get('regra_suavizacao_k', 'padrão')}` (mais estreito que a orla da placa menor) |",
        f"| Faixas de matiz | {faixas_txt} | Centro e largura medidos nas anotações da Seção 5.0, partindo das âncoras do CONTRAN |",
        f"| **Portas de saturação** | **{portas_txt}** | Estágio 0 da busca em grade, por descida em coordenadas |",
        (f"| Faixas descartadas | {', '.join(fora)} | O estágio 0 mediu que elas custam mais em falso positivo do que rendem em detecção |"
         if fora else
         "| Faixas descartadas | nenhuma | Todas as cores medidas na Seção 5.0 valeram a pena no estágio 0 |"),
        f"| **Kernel de abertura** | **{p['k_abertura']}×{p['k_abertura']}** | `{memoria.get('regra_k_abertura', 'padrão')}` |",
        f"| **Kernel de fechamento** | **{p['k_fechamento']}×{p['k_fechamento']}** | `{memoria.get('regra_k_fechamento', 'padrão')}` |",
        f"| **Método de limiarização** | **{escolha['metodo_escolhido']}** | {escolha['criterio']}. "
        f"Empate técnico entre {len(escolha['empate_tecnico_entre_metodos'])} métodos "
        f"({', '.join(escolha['empate_tecnico_entre_metodos'])}), resolvido pelo mais simples |",
        f"| Limiar T de Otsu (diagnóstico) | recalculado por imagem: média {desempenho['limiar_T_medio']}, "
        f"desvio {desempenho['limiar_T_desvio']}, de {desempenho['limiar_T_minimo']} a "
        f"{desempenho['limiar_T_maximo']} | Otsu restrito sobre o mapa de evidência, só como diagnóstico, "
        f"não entra na segmentação. O desvio mostra o quanto a "
        f"iluminação varia entre imagens. {desempenho['imagens_sem_evidencia_cromatica']} imagem(ns) sem "
        f"nenhum pixel com evidência de cor ficam fora da conta |",
        f"| Limiar do método global | {p['limiar_global']} | "
        f"{origem_interna('limiar_global', 'Corte único pra imagem toda')} |",
        f"| `blockSize` da adaptativa | {p['adapt_bloco']} px | "
        f"{origem_interna('adapt_bloco', 'Tamanho da janela usada pra calcular a média local')} |",
        f"| `C` da adaptativa | {p['adapt_c']} | "
        f"{origem_interna('adapt_c', texto_c)} |",
        f"| **Área mínima de contorno** | **{escolha['area_minima_px2']} px²** | Descarta o quantil {escolha['quantil_area_min']:.2f} inferior das placas anotadas (× 0,45 de preenchimento) |",
        (f"| **Área máxima de contorno** | **{escolha['area_maxima_px2']} px²** | "
         f"Descarta o quantil superior a {escolha['quantil_area_max']:.2f}: é um filtro de escala contra fachada e vegetação fotografadas de perto |"
         if escolha["area_maxima_px2"] else
         "| **Área máxima de contorno** | sem teto | A varredura do estágio 2 não mostrou ganho em limitar a área |"),
        f"| Razão de aspecto aceita | de {p['razao_aspecto'][0]} a {p['razao_aspecto'][1]} | Rejeita postes, faixas e meios-fios |",
        f"| Extensão mínima | {p['extensao_minima']} | Rejeita contorno rendilhado, normalmente vegetação |",
        f"| Solidez mínima | {p['solidez_minima']} | Toda placa normativa é convexa |",
        "",
        "### Dataset",
        "",
        f"- Fonte: {d['fonte']}",
        f"- {d['imagens']} imagens · {d['objetos_anotados']} objetos anotados · {d['classes']} classes",
        f"- Dimensão mediana: {d['dimensao_mediana']} px",
        "",
        "### Protocolo de avaliação",
        "",
        f"- {protocolo['amostra_ajuste']} imagens de **ajuste** (escolha dos parâmetros) e "
        f"{protocolo['amostra_validacao']} de **validação**, sorteadas com semente fixa",
        f"- Divisão {protocolo['divisao']['criterio'].replace('gravacao', 'gravação')}: "
        f"{protocolo['divisao']['trechos_ajuste']} trechos de ajuste e "
        f"{protocolo['divisao']['trechos_validacao']} de validação. O quadro de validação mais "
        f"próximo de um de ajuste está a {protocolo['divisao']['menor_distancia_ajuste_validacao_s']} s",
        f"- Casamento detecção ↔ anotação por IoU ≥ {desempenho['limiar_iou']}",
        "",
        "### Desempenho da linha de base clássica (amostra de validação retida)",
        "",
        f"- Precisão {desempenho['precisao']} · Recall {desempenho['recall']} · "
        f"F1 {desempenho['f1']} ({desempenho['imagens_avaliadas']} imagens)",
        f"- Erro absoluto médio de contagem: {desempenho['erro_absoluto_medio_contagem']} objetos por imagem "
        f"(chutar sempre {desempenho['chute_constante_de_contagem']} placa(s) dá "
        f"{desempenho['erro_absoluto_medio_chute_constante']})",
        f"- Contagem exata em {desempenho['imagens_com_contagem_exata']} de "
        f"{desempenho['imagens_avaliadas']} imagens",
    ]
    return "\n".join(linhas)


texto_md = tabela_markdown_parametros(REGISTRO)
(DIR_SAIDA / "parametros_adotados.md").write_text(texto_md, encoding="utf-8")
print(texto_md)

In [ ]:
GRADE.to_csv(DIR_SAIDA / "busca_em_grade.csv", index=False, encoding="utf-8")
COMPARACAO_LIMIAR.to_csv(DIR_SAIDA / "comparacao_limiarizacao.csv", encoding="utf-8")
ABLACAO.to_csv(DIR_SAIDA / "ablacao_preprocessamento.csv", index=False, encoding="utf-8")
AVALIACAO.to_csv(DIR_SAIDA / "avaliacao_por_imagem.csv", index=False, encoding="utf-8")
if not DISTRIBUICAO_FORMAS.empty:
    DISTRIBUICAO_FORMAS.to_csv(DIR_SAIDA / "distribuicao_formas.csv", encoding="utf-8")

pacote = RAIZ / "outputs_checkpoint1.zip"
with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
    for arquivo in sorted(DIR_SAIDA.rglob("*")):
        if arquivo.is_file():
            z.write(arquivo, arquivo.relative_to(RAIZ))
    diagrama = DIR_DOCS / "arquitetura_pipeline.png"
    if diagrama.exists():
        z.write(diagrama, diagrama.relative_to(RAIZ))

print("Artefatos gerados em outputs/:")
for arquivo in sorted(DIR_SAIDA.rglob("*")):
    if arquivo.is_file():
        print(f"  {arquivo.relative_to(DIR_SAIDA)}  ({arquivo.stat().st_size / 1024:.0f} KB)")
print(f"\nPacote: {pacote} ({pacote.stat().st_size / 1024 / 1024:.1f} MB)")

if EM_COLAB:
    from google.colab import files
    print("\nBaixe o pacote para versionar no repositorio:")
    files.download(str(pacote))

---
## 11. Limitações identificadas

O que este pipeline não resolve, e sobre o que a 2ª Etapa precisa ganhar.

1. **Falso positivo de mesma cromaticidade.** Lanterna, carro vermelho, toldo, solo exposto
   e grama seca têm matiz e saturação de placa. Os filtros de forma tiram parte, mas objeto
   convexo de proporção compatível é indistinguível por cor e geometria.

2. **Cada faixa tem um confundidor natural.** O verde disputa com vegetação, o azul com o
   céu, o amarelo com solo exposto, o vermelho com veículo. Por isso a porta de saturação e
   o descarte de faixa são decididos por métrica, não por hipótese.

3. **Placa pequena é o teto do recall.** Um décimo das placas tem menos de 19 px de lado
   equivalente, mesmo na resolução nativa, e foi por isso que a busca subiu o piso de área.
   Usar a resolução nativa em vez da esticada já ajudou (Seção 2.1), mas o limite agora é a
   distância do objeto na cena.

4. **Círculo e octógono sob perspectiva.** Indistinguíveis por descritor clássico, e o
   código marca `forma_ambigua = True` em vez de arbitrar.

5. **Desbotamento e oclusão.** Película desbotada cai abaixo da porta `s_min`, e oclusão
   fragmenta o contorno e derruba a solidez. Os dois geram falso negativo que nenhum ajuste
   de limiar recupera sem inundar a máscara.

6. **Não identifica o significado da placa.** O pipeline entrega forma e posição, não a
   categoria (A-1a, R-19). Essa é, por definição, a tarefa da 2ª Etapa.
